In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"       # before CUDA initialises (workers set it too)

import hashlib, json, time
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "dmrgt4").mkdir(parents=True, exist_ok=True)   # %%writefile cells below do not create folders
RUN_T0 = time.time()

# ─── Pre-registered configuration (spec §4) — fixed BEFORE any data is touched ───
CONFIG = {
    "phase": "4",
    "seeds": [0, 1, 2, 3, 4],
    "folds": "identical to Phase 3 (fold_1_2019_calm, fold_2_2020_covid, fold_3_2021, fold_4_2022_rates)",
    "graph": {"k_pos": 8, "k_neg": 4, "corr_window": 60, "file": "graphs_baseline.parquet"},
    "features": 8,
    "backtest": {"commission_bps": 10, "slippage_bps": 5, "decile_fraction": 0.2},

    "arms": {
        "A": {"encoder": "transformer", "lookback": 30, "adaptive_huber": False, "required": True},
        "B": {"encoder": "mamba",       "lookback": 30, "adaptive_huber": False, "required": True},
        "C": {"encoder": "mamba",       "lookback": 60, "adaptive_huber": False, "required": True},
        "D": {"encoder": "mamba",       "lookback": 60, "adaptive_huber": True,  "required": False},
    },

    "training": {
        "max_epochs": 50, "early_stop_patience": 10, "early_stop_metric": "val_mean_rank_ic",
        "batch_size_ladder": [32, 16, 8],          # same batch size for ALL arms
        "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-4,
        "warmup_epochs": 3, "warmup_start_lr": 1e-4,
        "plateau": {"mode": "max", "factor": 0.5, "patience": 4, "min_lr": 1e-5},
        "grad_clip_norm": 1.0,
        "huber_delta_fixed": 1.0,
        "adaptive_huber": {"c": 1.345, "min": 0.05, "max": 2.0},
        "retry_failed_job_once": True,
    },

    "mamba": {"d_model": 64, "n_layers": 2, "d_state": 16, "d_conv": 4, "expand": 2, "dropout": 0.1},

    "selection_4A": {"metric": "val_sharpe", "tiebreak_metric": "val_mean_rank_ic", "tie_threshold": 0.02,
                     "candidates": ["B", "C", "D"]},

    "phase4b": {
        "corr_window": 60,
        "target_rebuild_rates": [0.10, 0.20, 0.35, 0.50],
        "signals": ["corr_drift", "mamba_delta", "combined"],
        "max_staleness_days": 20,
        "full_recompute_every": 250,
        "val_noninferiority_margin_sharpe": 0.10,
        "test_noninferiority_margin_sharpe": 0.25,
        "scale_benchmark_n_stocks": [40, 100, 500],
    },

    "significance": {"bootstrap_iterations": 5000, "bootstrap_block_length": 10, "alpha": 0.05,
                     "secondary_correction": "holm"},

    "leakage_gate": {"max_abs_ic_in_standard_errors": 3.0},

    "budget": {"session_limit_hours": 12.0, "plan_limit_hours": 10.5,
               "expected_epochs_for_projection": 35, "safety_factor": 1.15,
               "phase4b_estimate_minutes": 45, "scoring_and_bench_estimate_minutes": 20},
}

# Static Phase 3 values that live in configs.yaml (not in the cache input), copied
# unchanged: universe, sectors, feature columns, walk-forward folds. Part of the snapshot hash.
STATIC = {
    "universe": [
        "AAPL",
        "MSFT",
        "NVDA",
        "AVGO",
        "CRM",
        "JPM",
        "BRK-B",
        "GS",
        "BAC",
        "V",
        "JNJ",
        "PFE",
        "UNH",
        "ABBV",
        "MRK",
        "XOM",
        "CVX",
        "COP",
        "SLB",
        "EOG",
        "AMZN",
        "HD",
        "NKE",
        "TSLA",
        "SBUX",
        "PG",
        "KO",
        "WMT",
        "COST",
        "PEP",
        "HON",
        "CAT",
        "BA",
        "UPS",
        "UNP",
        "NEE",
        "DUK",
        "SO",
        "AEP",
        "EXC"
    ],
    "sectors": {
        "AAPL": "Information Technology",
        "MSFT": "Information Technology",
        "NVDA": "Information Technology",
        "AVGO": "Information Technology",
        "CRM": "Information Technology",
        "JPM": "Financials",
        "BRK-B": "Financials",
        "GS": "Financials",
        "BAC": "Financials",
        "V": "Financials",
        "JNJ": "Health Care",
        "PFE": "Health Care",
        "UNH": "Health Care",
        "ABBV": "Health Care",
        "MRK": "Health Care",
        "XOM": "Energy",
        "CVX": "Energy",
        "COP": "Energy",
        "SLB": "Energy",
        "EOG": "Energy",
        "AMZN": "Consumer Discretionary",
        "HD": "Consumer Discretionary",
        "NKE": "Consumer Discretionary",
        "TSLA": "Consumer Discretionary",
        "SBUX": "Consumer Discretionary",
        "PG": "Consumer Staples",
        "KO": "Consumer Staples",
        "WMT": "Consumer Staples",
        "COST": "Consumer Staples",
        "PEP": "Consumer Staples",
        "HON": "Industrials",
        "CAT": "Industrials",
        "BA": "Industrials",
        "UPS": "Industrials",
        "UNP": "Industrials",
        "NEE": "Utilities",
        "DUK": "Utilities",
        "SO": "Utilities",
        "AEP": "Utilities",
        "EXC": "Utilities"
    },
    "feature_columns": [
        "f1_logret",
        "f2_price_ma",
        "f3_rsi",
        "f4_macd",
        "f5_atr",
        "f6_vol_zscore",
        "f7_market_ret",
        "f8_vix"
    ],
    "folds": [
        {
            "name": "fold_1_2019_calm",
            "train_start": "2015-01-01",
            "train_end": "2018-12-31",
            "test_start": "2019-01-01",
            "test_end": "2019-12-31"
        },
        {
            "name": "fold_2_2020_covid",
            "train_start": "2015-01-01",
            "train_end": "2019-12-31",
            "test_start": "2020-01-01",
            "test_end": "2020-12-31"
        },
        {
            "name": "fold_3_2021",
            "train_start": "2015-01-01",
            "train_end": "2020-12-31",
            "test_start": "2021-01-01",
            "test_end": "2021-12-31"
        },
        {
            "name": "fold_4_2022_rates",
            "train_start": "2015-01-01",
            "train_end": "2021-12-31",
            "test_start": "2022-01-01",
            "test_end": "2022-12-31"
        }
    ],
    "source": "values copied from Phase 3 configs.yaml (universe, sectors, features.columns, folds); not in the cache input"
}

SNAPSHOT = {"CONFIG": CONFIG, "STATIC": STATIC}
_snap_text = json.dumps(SNAPSHOT, indent=2, sort_keys=True)
(WORK / "config_snapshot.json").write_text(_snap_text)
CONFIG_SHA256 = hashlib.sha256(_snap_text.encode()).hexdigest()
print("config_snapshot.json sha256:", CONFIG_SHA256)


In [ ]:
# ─── Cell 1: environment + GPU checks (hard stop on failure) ───
import platform, sys
import numpy, pandas, scipy, torch

n_gpus = torch.cuda.device_count()
assert n_gpus >= 1, "No CUDA GPU visible — set the Kaggle accelerator to GPU T4 x2."
gpu_info = []
for i in range(n_gpus):
    cc = torch.cuda.get_device_capability(i)
    name = torch.cuda.get_device_name(i)
    gpu_info.append({"index": i, "name": name, "capability": list(cc)})
    assert cc >= (7, 0), (f"GPU {i} ({name}) has compute capability {cc[0]}.{cc[1]} < 7.0 "
                          f"(P100 is sm_60) — select GPU T4 x2.")
print(f"{n_gpus} GPU(s):", [g["name"] for g in gpu_info])
if n_gpus == 1:
    print("Only 1 GPU detected: the notebook runs sequentially; the budget planner accounts for it.")

run_manifest = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "numpy": numpy.__version__, "pandas": pandas.__version__,
    "scipy": scipy.__version__, "gpus": gpu_info, "n_gpus": n_gpus,
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "config_sha256": CONFIG_SHA256, "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(RUN_T0)),
}
(WORK / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))


In [ ]:
# ─── Cell 2: cache resolution + hash verification (hard stop on failure) ───
from pathlib import Path
import hashlib, json

CANDIDATES = [
    Path("/kaggle/input/dmrgt-phase-4-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-phase-4-cache"),
    Path("/kaggle/input/dmrgt-phase-3-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-phase-3-cache"),
]
def resolve_cache():
    for c in CANDIDATES:
        if (c / "features.parquet").exists():
            return c
    hits = list(Path("/kaggle/input").rglob("features.parquet"))
    if len(hits) == 1:
        return hits[0].parent
    raise FileNotFoundError(f"Cache not found or ambiguous: {hits}")

CACHE = resolve_cache()
for _req in ["prices.parquet", "features.parquet", "targets.parquet", "graphs_baseline.parquet", "hashes.json"]:
    assert (CACHE / _req).exists(), f"required cache file missing: {_req}"
expected = json.loads((CACHE / "hashes.json").read_text())
for name, h in expected.items():
    if not name.endswith(".parquet"):      # skip metadata keys such as generated_at
        continue
    actual = hashlib.sha256((CACHE / name).read_bytes()).hexdigest()
    assert actual == h, f"Hash mismatch for {name}"
os.environ["DMRGT4_CACHE"] = str(CACHE)    # inherited by the worker processes
print("cache OK:", CACHE, "—", sum(k.endswith('.parquet') for k in expected), "parquet hashes verified")


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_data.py
"""DMRGT Phase 4 — data module (spec §6.1).

Ported from Phase 3 (reference/phase3_src/): splits.py (make_folds, Fold),
train.assemble_fold (windows, label alignment, train-only scaler, graph-
coverage drop) and data.py (config access). Changes (the only two the spec
allows): (1) lookback is a parameter; (2) the test split is gated behind
the two lock files. Config comes from config_snapshot.json (written by
notebook Cell 0) because configs.yaml is not part of the cache input.
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

WORK = Path(os.environ.get("DMRGT4_WORK", "/kaggle/working"))
LOCKS = [WORK / "phase4_winner.json", WORK / "phase4b_policy.json"]

CANDIDATES = [
    Path("/kaggle/input/dmrgt-phase-4-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-phase-4-cache"),
    Path("/kaggle/input/dmrgt-phase-3-cache"),
    Path("/kaggle/input/datasets/harishsivakumar1912/dmrgt-phase-3-cache"),
]


def resolve_cache() -> Path:
    env = os.environ.get("DMRGT4_CACHE")
    if env:
        return Path(env)
    for c in CANDIDATES:
        if (c / "features.parquet").exists():
            return c
    hits = list(Path("/kaggle/input").rglob("features.parquet"))
    if len(hits) == 1:
        return hits[0].parent
    raise FileNotFoundError(f"Cache not found or ambiguous: {hits}")


# ─── config (from the Cell 0 snapshot) ─────────────────────────────────────

def load_snapshot() -> dict:
    return json.loads((WORK / "config_snapshot.json").read_text())


def load_cfg() -> dict:
    """Phase-3-shaped cfg dict (universe / features / graph / training /
    backtest / significance / folds) assembled from the snapshot, so the
    ported eval / stats / fold code runs unchanged."""
    snap = load_snapshot()
    C, S = snap["CONFIG"], snap["STATIC"]
    sig = C["significance"]
    return {
        "universe": {"tickers": list(S["universe"])},
        "sectors": dict(S["sectors"]),
        "features": {"lookback_days": 30, "columns": list(S["feature_columns"])},
        "graph": {"corr_window": C["graph"]["corr_window"], "k_pos": C["graph"]["k_pos"],
                  "k_neg": C["graph"]["k_neg"], "symmetrise": False,
                  "include_sector": True, "min_history_days": C["graph"]["corr_window"] + 1},
        "training": {"huber_delta": C["training"]["huber_delta_fixed"]},
        "backtest": dict(C["backtest"]),
        "significance": {"wilcoxon_alpha": sig["alpha"],
                         "bootstrap_iterations": sig["bootstrap_iterations"],
                         "bootstrap_block_length": sig["bootstrap_block_length"]},
        "folds": list(S["folds"]),
    }


# ─── folds (ported verbatim from splits.py) ────────────────────────────────

@dataclass
class Fold:
    """One walk-forward fold with train_dates, val_dates, test_dates."""
    name: str
    fold_idx: int
    train_dates: pd.DatetimeIndex
    val_dates: pd.DatetimeIndex
    test_dates: pd.DatetimeIndex

    def __repr__(self) -> str:
        def span(idx: pd.DatetimeIndex) -> str:
            if len(idx) == 0:
                return "empty"
            return f"{idx[0].date()}→{idx[-1].date()} (n={len(idx)})"
        return (f"Fold({self.name!r}, train={span(self.train_dates)}, "
                f"val={span(self.val_dates)}, test={span(self.test_dates)})")


def make_folds(
    cfg: dict,
    available_dates: Iterable[pd.Timestamp] | None = None,
    val_fraction: float = 0.15,
) -> list[Fold]:
    """Build walk-forward folds from configs.yaml.

    `available_dates` restricts each configured window to dates that
    actually exist in the cache (the features table's calendar). The
    latest val_fraction of each training window becomes validation.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")

    avail = (pd.DatetimeIndex(sorted(set(pd.Timestamp(d) for d in available_dates)))
             if available_dates is not None else None)
    folds_cfg = cfg.get("folds", [])
    if not folds_cfg:
        raise ValueError("configs.yaml has no 'folds' section")

    out: list[Fold] = []
    for i, fc in enumerate(folds_cfg):
        name = str(fc.get("name", f"fold_{i}"))
        train_start = pd.Timestamp(fc["train_start"])
        train_end = pd.Timestamp(fc["train_end"])
        test_start = pd.Timestamp(fc["test_start"])
        test_end = pd.Timestamp(fc["test_end"])

        # ── hard chronological invariant (Rule 2) ─────────────────────
        if train_end >= test_start:
            raise AssertionError(
                f"Fold {name!r}: train_end ({train_end.date()}) must be "
                f"strictly before test_start ({test_start.date()}) — "
                f"otherwise the model sees the test window during training."
            )

        # ── build the full training and test index ─────────────────────
        if avail is not None:
            # Configured window, intersected with dates the cache has.
            # The boundary itself is owned by the config; the data
            # restricts what we can actually use.
            train_mask = (avail >= train_start) & (avail <= train_end)
            test_mask = (avail >= test_start) & (avail <= test_end)
            train_full = avail[train_mask]
            test_idx = avail[test_mask]
        else:
            # No availability constraint (tests / running splits without
            # the cache): synthesise business-day dates. The next consumer
            # that loads the features table intersects again.
            train_full = pd.bdate_range(train_start, train_end)
            test_idx = pd.bdate_range(test_start, test_end)

        if len(train_full) == 0:
            raise ValueError(f"Fold {name!r}: no train dates after intersecting with available dates")
        if len(test_idx) == 0:
            raise ValueError(f"Fold {name!r}: no test dates after intersecting with available dates")

        # ── split off the validation portion ───────────────────────────
        # The latest val_fraction of the training window becomes the
        # validation set. Chronological, not random.
        n_train_full = len(train_full)
        n_val = int(round(n_train_full * val_fraction))
        if n_val <= 0 or n_val >= n_train_full:
            raise ValueError(
                f"Fold {name!r}: val_fraction={val_fraction} produced "
                f"n_val={n_val} from n_train_full={n_train_full}; "
                f"need at least 1 train and 1 val date"
            )
        train_idx = train_full[:-n_val]
        val_idx = train_full[-n_val:]

        # ── zero-overlap asserts — fail loudly, never silently ─────────
        assert len(set(train_idx).intersection(val_idx)) == 0
        assert len(set(train_idx).intersection(test_idx)) == 0
        assert len(set(val_idx).intersection(test_idx)) == 0
        # Chronological: last train date precedes first val date, which
        # precedes the first test date.
        assert train_idx.max() < val_idx.min()
        assert val_idx.max() < test_idx.min()
        assert train_idx.max() < test_idx.min()

        out.append(Fold(
            name=name,
            fold_idx=i,
            train_dates=train_idx,
            val_dates=val_idx,
            test_dates=test_idx,
        ))

    return out


# ─── test access gate (Change 2) ───────────────────────────────────────────

def test_unlocked() -> bool:
    return all(p.exists() for p in LOCKS)


def assert_test_unlocked() -> None:
    if not test_unlocked():
        raise PermissionError(
            "Test split is locked until phase4_winner.json and phase4b_policy.json exist.")


# ─── tables, loaded once per process ───────────────────────────────────────

_TABLES: dict = {}


def tables() -> dict:
    """features / targets pivots + folds for this process (cached)."""
    if _TABLES:
        return _TABLES
    cache = resolve_cache()
    cfg = load_cfg()
    features = pd.read_parquet(cache / "features.parquet")
    targets = pd.read_parquet(cache / "targets.parquet")
    feature_cols = list(cfg["features"]["columns"])
    missing = [c for c in feature_cols if c not in features.columns]
    assert not missing, f"config lists feature columns absent from features.parquet: {missing}"
    assert len(feature_cols) == 8, f"Expected 8 features, got {len(feature_cols)}"
    feat_wide = (features.pivot(index="date", columns="ticker", values=feature_cols).sort_index())
    # Fix the feature level to config order (pivot sorts lexicographically).
    feat_wide = feat_wide.reindex(columns=pd.MultiIndex.from_product(
        [feature_cols, list(feat_wide.columns.get_level_values("ticker").unique())],
        names=["feature", "ticker"]))
    tgt_wide = (targets.pivot(index="date", columns="ticker", values="excess_return").sort_index())
    folds = make_folds(cfg, available_dates=features["date"].unique())
    _TABLES.update(cfg=cfg, cache=cache, features=features, targets=targets,
                   feat_wide=feat_wide, tgt_wide=tgt_wide, folds=folds,
                   feature_cols=feature_cols)
    return _TABLES


def fold_by_name(name: str) -> Fold:
    for f in tables()["folds"]:
        if f.name == name:
            return f
    raise KeyError(name)


@dataclass
class SplitData:
    name: str
    split: str
    X: np.ndarray            # [n_days, n_stocks, lookback, n_features], scaled
    y: np.ndarray            # [n_days, n_stocks]
    dates: np.ndarray
    tickers: list


class FoldBuilder:
    """Per-fold, per-lookback tensors. Train and val are built eagerly;
    test is built lazily and only after both lock files exist."""

    def __init__(self, fold: Fold, lookback: int, graph_dates=None):
        T = tables()
        self.fold, self.lookback, self.graph_dates = fold, lookback, graph_dates
        self.cfg = T["cfg"]
        self.universe = list(self.cfg["universe"]["tickers"])
        self.feature_cols = T["feature_cols"]
        self.feat_wide, self.tgt_wide = T["feat_wide"], T["tgt_wide"]
        self.dropped_no_graph = {"train": [], "val": [], "test": []}

        X_tr, y_tr, d_tr = self._build(fold.train_dates, "train")
        X_va, y_va, d_va = self._build(fold.val_dates, "val")
        assert len(X_tr) > 0, f"{fold.name}: no training windows at lookback {lookback}"

        # ── bounded lookback drop (Change 1) ──────────────────────────
        # Dates Phase 3's 30-day windows kept that this lookback loses.
        locs = [self.feat_wide.index.get_loc(d) for d in fold.train_dates
                if d in self.feat_wide.index]
        base = sum(1 for l in locs if l >= 30 - 1)
        now = sum(1 for l in locs if l >= lookback - 1)
        self.lookback_dropped = base - now
        self.lookback_drop_frac = self.lookback_dropped / max(1, base)
        assert self.lookback_drop_frac < 0.05, (
            f"{fold.name}: lookback {lookback} drops {self.lookback_dropped}/{base} "
            f"train days ({self.lookback_drop_frac:.2%}) — over the 5% bound")
        print(f"[d4_data] {fold.name} lookback={lookback}: dropped {self.lookback_dropped}/{base} "
              f"train days vs the 30-day window ({self.lookback_drop_frac:.2%})", flush=True)

        # ── bounded graph-coverage check (ported from Phase 3 §3.5) ───
        total_dropped = sum(len(v) for v in self.dropped_no_graph.values())
        total_dates = len(fold.train_dates) + len(fold.val_dates)
        if graph_dates is not None and total_dropped > 0:
            drop_frac = total_dropped / total_dates
            assert drop_frac < 0.05, (
                f"{total_dropped} dates ({drop_frac:.2%}) missing graph coverage — "
                f"check which graph file this run was pointed at.")

        # ── scaler fitted on training dates ONLY ──────────────────────
        n_feat = X_tr.shape[3]
        scaler = StandardScaler().fit(X_tr.reshape(-1, n_feat))
        assert scaler.mean_.shape == (n_feat,)
        self._scaler = scaler
        self.scaler_mean = scaler.mean_.astype(float)
        self.scaler_scale = scaler.scale_.astype(float)
        self.scaler_fit_dates = (pd.Timestamp(d_tr[0]), pd.Timestamp(d_tr[-1]))
        self._splits = {
            "train": SplitData(fold.name, "train", self._transform(X_tr), y_tr, d_tr, self.universe),
            "val": SplitData(fold.name, "val", self._transform(X_va), y_va, d_va, self.universe),
        }
        assert pd.Timestamp(d_tr.max()) < pd.Timestamp(d_va.min()), "train/val chronology violated"

    def _transform(self, X):
        if len(X) == 0:
            return X
        shape = X.shape
        return self._scaler.transform(X.reshape(-1, shape[3])).reshape(shape)

    def _build(self, date_set, which: str):
        lookback, feat_wide, tgt_wide = self.lookback, self.feat_wide, self.tgt_wide
        universe = self.universe
        Xs, ys, kept = [], [], []
        for d in date_set:
            if d not in feat_wide.index:
                continue
            if d not in tgt_wide.index:
                continue
            loc = feat_wide.index.get_loc(d)
            start = loc - lookback + 1
            if start < 0:
                continue  # not enough history — drop, never zero-pad
            if self.graph_dates is not None and d not in self.graph_dates:
                self.dropped_no_graph[which].append(d)
                continue
            window_dates = feat_wide.index[start: loc + 1]
            assert len(window_dates) == lookback
            assert window_dates[-1] == d, "window last date != prediction date"
            X = np.empty((len(universe), lookback, len(self.feature_cols)), dtype=np.float64)
            for j, t in enumerate(universe):
                X[j] = feat_wide.loc[window_dates, (slice(None), t)].values
            Xs.append(X)
            kept.append(d)
            ys.append(tgt_wide.loc[d, universe].values)
        X = (np.stack(Xs, axis=0) if Xs
             else np.empty((0, len(universe), lookback, len(self.feature_cols))))
        y = np.stack(ys, axis=0) if ys else np.empty((0, len(universe)))
        return X, y, np.array(kept)

    def get(self, split: str) -> SplitData:
        if split == "test":
            assert_test_unlocked()
            if "test" not in self._splits:
                X, y, d = self._build(self.fold.test_dates, "test")
                assert len(X) > 0
                assert pd.Timestamp(self._splits["val"].dates.max()) < pd.Timestamp(d.min()), \
                    "val/test chronology violated"
                self._splits["test"] = SplitData(self.fold.name, "test", self._transform(X), y, d,
                                                 self.universe)
        return self._splits[split]


_BUILDERS: dict = {}


def get_builder(fold_name: str, lookback: int = 30, graph_dates=None) -> FoldBuilder:
    key = (fold_name, lookback)
    if key not in _BUILDERS:
        _BUILDERS[key] = FoldBuilder(fold_by_name(fold_name), lookback, graph_dates)
    return _BUILDERS[key]


def get_split(fold, split, lookback: int = 30, graph_dates=None) -> SplitData:
    """The only way tensors leave this module. split='test' raises
    PermissionError until phase4_winner.json and phase4b_policy.json exist."""
    if split == "test":
        assert_test_unlocked()
    name = fold if isinstance(fold, str) else fold.name
    return get_builder(name, lookback, graph_dates).get(split)


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_graph.py
"""DMRGT Phase 4 — graph module (spec §6.2).

Ported verbatim from Phase 3 graph.py: _pct_change_no_fill, _universe_returns,
_sector_edges, and the correlation→edges rule (nan_to_num, diagonal ∓inf,
argsort, top-k_pos / bottom-k_neg, sector edges, directed). The daily-edge
loader reads graphs_baseline.parquet.

New: RollingCorr (incremental float64 correlation), run_policy (trigger-
policy runner), trigger rules + threshold calibration, and the Phase 4B
inference engine (checkpoint → per-day encodings/Δ → policy graphs →
predictions). Inference only; nothing here trains.
"""

from __future__ import annotations

import time
from collections import Counter, deque

import numpy as np
import pandas as pd
import torch

import d4_data
import d4_eval
import d4_models


# ─── ported from Phase 3 graph.py ──────────────────────────────────────────

def _pct_change_no_fill(df: pd.DataFrame) -> pd.DataFrame:
    """pct_change with NO filling, across pandas 2.x and 3.x.

    pandas 2.x: pct_change(fill_method=None) — explicit no-fill.
    pandas 3.x: the fill_method keyword was removed BECAUSE no-fill
    became the only behaviour, so plain pct_change() is identical.
    The cache was built under pandas 2.x no-fill semantics; the §6b
    baseline-identity test proves the rebuild reproduces it byte-for-
    byte under either version.
    """
    try:
        return df.pct_change(fill_method=None)
    except TypeError:  # pandas ≥ 3: fill_method removed
        return df.pct_change()


def _universe_returns(prices: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Wide close prices for the 40 universe tickers → daily returns.

    rets.loc[d] is the return from d-1 to d; the first row (NaN) is
    dropped. Same op order as Phase 2 so the skip logic below lands on
    the same dates.
    """
    universe = list(cfg["universe"]["tickers"])
    px_wide = (prices.pivot(index="date", columns="ticker", values="close")
               .sort_index())
    missing = [t for t in universe if t not in px_wide.columns]
    if missing:
        raise ValueError(f"prices.parquet is missing universe tickers: {missing}")
    px_universe = px_wide[universe]
    rets = _pct_change_no_fill(px_universe)
    return rets.iloc[1:]


def _sector_edges(cfg: dict) -> tuple[list[int], list[int]]:
    """Static intra-sector edges (both directions), as index pairs.

    With 5 tickers per sector × 8 sectors: 8 × 20 = 160 directed edges.
    """
    universe = list(cfg["universe"]["tickers"])
    sectors_cfg = cfg.get("sectors", {})
    if not sectors_cfg:
        raise ValueError("configs.yaml missing 'sectors:' block — required for graph.py")
    include_sector = bool(cfg["graph"].get("include_sector", True))
    srcs: list[int] = []
    dsts: list[int] = []
    if not include_sector:
        return srcs, dsts
    n = len(universe)
    for i in range(n):
        for j in range(n):
            if i >= j:
                continue
            if sectors_cfg.get(universe[i]) == sectors_cfg.get(universe[j]):
                srcs.extend([i, j])  # directed both ways
                dsts.extend([j, i])
    sector_counts = Counter(sectors_cfg[t] for t in universe)
    expected = sum(c * (c - 1) for c in sector_counts.values())
    assert len(srcs) == expected, (
        f"Sector edge count mismatch: built {len(srcs)}, expected "
        f"{expected} from configs.yaml (5 per sector × 8 sectors → 160)"
    )
    return srcs, dsts



def edges_from_corr(corr: np.ndarray, cfg: dict, sector_src, sector_dst):
    """Correlation matrix → (edge_index [2, E], edge_type [E]) torch tensors.

    The rule of Phase 3's _edges_for_returns_window, unchanged: nan_to_num →
    diagonal −inf → argsort(−corr) for type 0 (top k_pos) → diagonal +inf →
    argsort(corr) for type 1 (bottom k_neg) → static sector edges (type 2).
    Rows are emitted in the cached file's (edge_type, src, dst) order.
    """
    n_stocks = corr.shape[0]
    k_pos = int(cfg["graph"]["k_pos"])
    k_neg = int(cfg["graph"]["k_neg"])
    corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0)   # returns a copy
    src_l, dst_l, typ_l = [], [], []
    np.fill_diagonal(corr, -np.inf)
    pos_order = np.argsort(-corr, axis=1)
    for i in range(n_stocks):
        for rank in range(k_pos):
            src_l.append(i); dst_l.append(int(pos_order[i, rank])); typ_l.append(0)
    np.fill_diagonal(corr, np.inf)
    neg_order = np.argsort(corr, axis=1)
    for i in range(n_stocks):
        for rank in range(k_neg):
            src_l.append(i); dst_l.append(int(neg_order[i, rank])); typ_l.append(1)
    for s, d in zip(sector_src, sector_dst):
        src_l.append(int(s)); dst_l.append(int(d)); typ_l.append(2)
    src_a, dst_a, typ_a = np.array(src_l), np.array(dst_l), np.array(typ_l)
    order = np.lexsort((dst_a, src_a, typ_a))            # (edge_type, src, dst)
    ei = torch.tensor(np.stack([src_a[order], dst_a[order]]), dtype=torch.long)
    et = torch.tensor(typ_a[order], dtype=torch.long)
    return ei, et


def make_graph_builder(cfg: dict):
    sector_src, sector_dst = _sector_edges(cfg)

    def build_graph(C):
        return edges_from_corr(C, cfg, sector_src, sector_dst)
    return build_graph


_EDGES: dict = {}


def load_daily_edges() -> dict:
    """graphs_baseline.parquet → {Timestamp: (edge_index, edge_type)}.
    Row order within a day is the file's order (what Phase 3 used)."""
    if not _EDGES:
        g = pd.read_parquet(d4_data.resolve_cache() / "graphs_baseline.parquet")
        for d, day in g.groupby("date", sort=True):
            ei = torch.tensor(day[["src", "dst"]].values.T, dtype=torch.long)
            et = torch.tensor(day["edge_type"].values, dtype=torch.long)
            _EDGES[pd.Timestamp(d)] = (ei, et)
    return _EDGES


def universe_returns(cfg: dict) -> pd.DataFrame:
    prices = pd.read_parquet(d4_data.resolve_cache() / "prices.parquet")
    return _universe_returns(prices, cfg)


# ─── incremental rolling correlation (new) ─────────────────────────────────

class RollingCorr:
    def __init__(self, window, n_stocks, full_recompute_every=250):
        self.W = window
        self.buf = deque()
        self.S1 = np.zeros(n_stocks)
        self.S2 = np.zeros((n_stocks, n_stocks))
        self.k = full_recompute_every
        self.n_updates = 0

    def push(self, r):                       # r: (N,) float64 returns for one day
        assert np.isfinite(r).all()
        self.buf.append(r)
        self.S1 += r
        self.S2 += np.outer(r, r)
        if len(self.buf) > self.W:
            old = self.buf.popleft()
            self.S1 -= old
            self.S2 -= np.outer(old, old)
        self.n_updates += 1
        if self.n_updates % self.k == 0:     # remove floating-point drift
            X = np.asarray(self.buf)
            self.S1, self.S2 = X.sum(0), X.T @ X

    def corr(self):
        W = len(self.buf)
        cov = (self.S2 - np.outer(self.S1, self.S1) / W) / (W - 1)
        d = np.sqrt(np.diag(cov))
        return cov / np.outer(d, d)


def warm_rolling_corr(rets: pd.DataFrame, end_date, window: int, full_recompute_every=250) -> RollingCorr:
    """RollingCorr holding the `window` returns ending at `end_date` inclusive."""
    loc = rets.index.get_loc(pd.Timestamp(end_date))
    assert loc >= window - 1, f"only {loc + 1} returns up to {end_date}"
    rc = RollingCorr(window, rets.shape[1], full_recompute_every)
    for r in rets.iloc[loc - window + 1: loc + 1].to_numpy(dtype=np.float64):
        rc.push(r)
    return rc


class PrecomputedCorr:
    """Replays correlation matrices computed once (calibration sims). Same
    push()/corr() interface as RollingCorr, so run_policy is one code path."""

    def __init__(self, C_by_step):
        self.C = C_by_step          # list: [C before first push, C after push 1, ...]
        self.i = 0

    def push(self, r):
        self.i += 1

    def corr(self):
        return self.C[self.i]


def corr_by_date(rets: pd.DataFrame, window: int) -> dict:
    """{date: corr matrix over the trailing window ending that date} for every
    date with a full window, via one incremental pass."""
    rc = RollingCorr(window, rets.shape[1])
    out = {}
    for i, (d, r) in enumerate(zip(rets.index, rets.to_numpy(dtype=np.float64))):
        rc.push(r)
        if i >= window - 1:
            out[pd.Timestamp(d)] = rc.corr()
    return out


# ─── trigger policy runner (new; inference only) ───────────────────────────

def run_policy(days, returns_for_day, build_graph, signal_for_day, fire_rule, max_stale, rc):
    """
    For each day: update correlations incrementally, compute signals using only
    information available that day, and rebuild the graph only if the rule fires
    (or the graph is older than max_stale days). Returns the graph used on each day
    and the list of rebuild days.
    """
    C_ref = rc.corr()
    graph = build_graph(C_ref)
    last, graphs, rebuilds = 0, {}, []
    for i, d in enumerate(days):
        rc.push(returns_for_day(d))          # alignment ported exactly from Phase 3 graph.py
        C = rc.corr()
        s_corr = np.linalg.norm(C - C_ref) / np.linalg.norm(C_ref)
        s_delta = signal_for_day(d)          # Mamba Δ for day d, computed before message passing
        if fire_rule(s_corr, s_delta) or (i - last) >= max_stale:
            graph, C_ref, last = build_graph(C), C, i
            rebuilds.append(d)
        graphs[d] = graph
    return graphs, rebuilds


# daily = rule always fires; frozen_at_start = rule never fires and max_stale = ∞.
def rule_daily(s_corr, s_delta):
    return True


def rule_never(s_corr, s_delta):
    return False


def make_rule(signal: str, tau_corr: float | None, tau_delta: float | None):
    if signal == "corr_drift":
        return lambda sc, sd: sc > tau_corr
    if signal == "mamba_delta":
        return lambda sc, sd: sd > tau_delta
    if signal == "combined":
        return lambda sc, sd: (sc > tau_corr) or (sd > tau_delta)
    raise ValueError(signal)


# ─── threshold calibration on TRAINING days (leakage-free) ─────────────────

def simulate_rate(sim_days, C_steps, delta_by_day, rule, max_stale) -> float:
    """Realised rebuild rate of `rule` over sim_days, using the SAME
    run_policy as inference (graph building stubbed out)."""
    pc = PrecomputedCorr(C_steps)
    zero = np.zeros(1)
    _, rebuilds = run_policy(
        sim_days, lambda d: zero, lambda C: None,
        lambda d: float(delta_by_day[d]) if delta_by_day is not None else 0.0,
        rule, max_stale, pc)
    return len(rebuilds) / max(1, len(sim_days))


def _bisect(rate_fn, lo, hi, target, iters=20):
    """Threshold whose rate is closest to target (rate is non-increasing in
    the threshold)."""
    best_t, best_err = hi, abs(rate_fn(hi) - target)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        r = rate_fn(mid)
        err = abs(r - target)
        if err < best_err:
            best_t, best_err = mid, err
        if r > target:
            lo = mid
        else:
            hi = mid
    return best_t


def calibrate_thresholds(signal, target_rate, sim_days, C_steps, delta_by_day, max_stale,
                         iters=20, corr_cache=None):
    """{'tau_corr', 'tau_delta', 'sim_rate'} for one (fold, model, signal, target).
    `corr_cache` memoises the corr-drift threshold per rate: it depends only on
    the fold's correlation history, not on the model."""
    corr_cache = {} if corr_cache is None else corr_cache
    d_vals = np.array([delta_by_day[d] for d in sim_days]) if delta_by_day is not None else None

    def corr_tau(rate):
        key = round(float(rate), 9)
        if key not in corr_cache:
            corr_cache[key] = _corr_tau(rate)
        return corr_cache[key]

    def _corr_tau(rate):
        f = lambda t: simulate_rate(sim_days, C_steps, None, make_rule("corr_drift", t, None), max_stale)
        return _bisect(f, 0.0, 5.0, rate, iters)

    def delta_tau(rate):
        f = lambda t: simulate_rate(sim_days, C_steps, delta_by_day, make_rule("mamba_delta", None, t), max_stale)
        return _bisect(f, float(d_vals.min()) - 1e-9, float(d_vals.max()) + 1e-9, rate, iters)

    if signal == "corr_drift":
        tc, td = corr_tau(target_rate), None
        rule = make_rule(signal, tc, None)
    elif signal == "mamba_delta":
        tc, td = None, delta_tau(target_rate)
        rule = make_rule(signal, None, td)
    elif signal == "combined":
        # One shared per-signal rate q; each signal's threshold is its own
        # single-signal calibration at q; bisect q so the UNION hits target.
        cache = {}

        def pair(q):
            if q not in cache:
                cache[q] = (corr_tau(q), delta_tau(q))
            return cache[q]

        def union_rate(q):
            tc_, td_ = pair(q)
            return simulate_rate(sim_days, C_steps, delta_by_day, make_rule("combined", tc_, td_), max_stale)

        # union rate rises with q, _bisect expects a rate that falls with its argument → search s = 1 − q
        s_ = _bisect(lambda x: union_rate(round(1.0 - x, 9)), 0.0, 1.0, target_rate, iters)
        q = round(1.0 - s_, 9)
        tc, td = pair(q)
        rule = make_rule(signal, tc, td)
    else:
        raise ValueError(signal)
    return {"tau_corr": None if tc is None else float(tc),
            "tau_delta": None if td is None else float(td),
            "sim_rate": float(simulate_rate(sim_days, C_steps, delta_by_day, rule, max_stale))}


# ─── Phase 4B inference engine ─────────────────────────────────────────────

def get_split_g(fold_name: str, split: str, lookback: int):
    """Split tensors with Phase 3's graph-coverage rule applied against the
    cached daily graph — the same call training uses, so datasets match."""
    gd = set(load_daily_edges().keys())
    return d4_data.get_split(fold_name, split, lookback=lookback, graph_dates=gd)


_RET_ROWS: dict = {}


def return_rows(rets: pd.DataFrame) -> dict:
    if not _RET_ROWS:
        for d, r in zip(rets.index, rets.to_numpy(dtype=np.float64)):
            _RET_ROWS[pd.Timestamp(d)] = r
    return _RET_ROWS


class ModelInference:
    """A trained checkpoint put to inference use: encodings and Δ per day are
    computed once; any graph policy then only needs the cheap propagate step."""

    def __init__(self, ckpt_path, device, snapshot: dict):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        self.ck = ck
        self.arm, self.fold, self.seed = ck["arm"], ck["fold"], int(ck["seed"])
        self.lookback, self.encoder_type = int(ck["lookback"]), ck["encoder"]
        self.device = device
        self.model = d4_models.make_model(self.encoder_type, snapshot["CONFIG"]["mamba"])
        self.model.load_state_dict(ck["model_state_dict"])
        self.model.to(device).eval()
        self._enc: dict = {}                 # split -> (h [n,N,d], delta [n] | None)

    def encode(self, split: str, batch: int = 64):
        if split in self._enc:
            return self._enc[split]
        sd = get_split_g(self.fold, split, self.lookback)
        X = torch.from_numpy(sd.X).float()
        hs, ds = [], []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                xb = X[i: i + batch].to(self.device)
                hs.append(self.model.encode(xb))
                if self.encoder_type == "mamba":
                    # signal read after encode, before propagate (spec §12)
                    ds.append(self.model.encoder.delta_per_day(len(xb)).cpu())
        h = torch.cat(hs, 0)
        delta = torch.cat(ds, 0).numpy().astype(np.float64) if ds else None
        self._enc[split] = (h, delta)
        return self._enc[split]

    def split_data(self, split: str):
        return get_split_g(self.fold, split, self.lookback)

    def delta_by_day(self, splits) -> dict | None:
        out = {}
        for s in splits:
            _, delta = self.encode(s)
            if delta is None:
                return None
            for d, v in zip(self.split_data(s).dates, delta):
                out[pd.Timestamp(d)] = float(v)
        return out

    def predict_with_graphs(self, split: str, graphs: dict, batch: int = 64) -> np.ndarray:
        """graphs: {date: (ei, et)} covering the split's sample dates."""
        h, _ = self.encode(split)
        dates = [pd.Timestamp(d) for d in self.split_data(split).dates]
        preds = []
        with torch.no_grad():
            for i in range(0, len(dates), batch):
                dd = dates[i: i + batch]
                ei = [graphs[d][0].to(self.device) for d in dd]
                et = [graphs[d][1].to(self.device) for d in dd]
                preds.append(self.model.propagate_with_edge_lists(h[i: i + len(dd)], ei, et).cpu().numpy())
        return np.concatenate(preds, 0)

    def frame(self, model_name: str, split: str, preds: np.ndarray) -> pd.DataFrame:
        sd = self.split_data(split)
        return d4_eval.make_pred_frame(model_name, self.fold, self.seed, split, sd.dates,
                                       sd.tickers, preds, sd.y)


def policy_days(rets: pd.DataFrame, fold, end_date) -> list:
    anchor = pd.Timestamp(fold.train_dates[-1])
    idx = rets.index
    return [pd.Timestamp(d) for d in idx[(idx > anchor) & (idx <= pd.Timestamp(end_date))]]


def run_named_policy(policy: dict, thresholds: dict | None, fold, rets: pd.DataFrame,
                     end_date, delta_by_day: dict | None, cfg: dict, window: int,
                     max_stale: float):
    """One policy over (anchor = last training day, end_date]. Returns
    (graphs, rebuild_days, stats). `policy`: {'kind': 'daily'|'frozen'|'trigger',
    'signal': ...}; `thresholds`: {'tau_corr','tau_delta'} for trigger kinds."""
    build_graph = make_graph_builder(cfg)
    build_time = [0.0]

    def timed_build(C):
        t0 = time.perf_counter()
        g = build_graph(C)
        build_time[0] += time.perf_counter() - t0
        return g

    kind = policy["kind"]
    if kind == "daily":
        rule, stale = rule_daily, max_stale
    elif kind == "frozen":
        rule, stale = rule_never, float("inf")
    else:
        rule = make_rule(policy["signal"], thresholds["tau_corr"], thresholds["tau_delta"])
        stale = max_stale
    days = policy_days(rets, fold, end_date)
    rows = return_rows(rets)
    rc = warm_rolling_corr(rets, fold.train_dates[-1], window)
    sig = (lambda d: delta_by_day.get(d, 0.0)) if delta_by_day is not None else (lambda d: 0.0)
    t0 = time.perf_counter()
    graphs, rebuilds = run_policy(days, lambda d: rows[d], timed_build, sig, rule, stale, rc)
    total = time.perf_counter() - t0
    return graphs, rebuilds, {"n_days": len(days), "n_rebuilds": len(rebuilds),
                              "rebuild_rate": len(rebuilds) / max(1, len(days)),
                              "graph_build_seconds": build_time[0], "policy_seconds": total}


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_models.py
"""DMRGT Phase 4 — models (spec §6.3).

Ported verbatim from Phase 3 model.py: PositionalEncoding, TransformerEncoder
(arm A), RelationMessagePassing (hand-rolled R-GCN), stack_per_sample_edges,
and the concat prediction head. New: MambaEncoder (pure PyTorch), and the
split forward (encode / propagate) on DMRGT so the Phase 4B graph decision
can happen between the two steps.
"""

from __future__ import annotations

import math
from collections.abc import Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al. 2017)."""

    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        divterm = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * divterm)
        pe[:, 1::2] = torch.cos(position * divterm)
        # [1, max_len, d_model] so it broadcasts against [B, T, D]
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        return x + self.pe[:, : x.size(1), :]


class TransformerEncoder(nn.Module):
    """Temporal encoder: 2 layers, 4 heads, causal mask over the
    lookback axis, 8 → 64. Byte-identical to Phase 1/2 — every trained
    Phase 3 model reuses this exact backbone."""

    def __init__(
        self,
        n_features: int = 8,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dim_ff: int = 256,
        dropout: float = 0.1,
        max_len: int = 64,
    ) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers

        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len)
        self.input_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        """X: [B, n_stocks, lookback, n_features] → [B, n_stocks, d_model].

        Each stock's lookback window is encoded independently along the
        time axis with a causal mask (position t sees positions ≤ t);
        the last hidden state is the stock's embedding.
        """
        if X.dim() != 4:
            raise ValueError(
                f"encode() expects 4D input [B, n_stocks, L, F], "
                f"got {tuple(X.shape)}"
            )
        B, n_stocks, lookback, n_features = X.shape
        assert n_features == self.n_features, (
            f"Model built for n_features={self.n_features} but got "
            f"n_features={n_features}"
        )
        h = X.reshape(B * n_stocks, lookback, n_features)
        h = self.input_proj(h)
        h = self.pos_enc(h)
        h = self.input_dropout(h)

        causal_mask = torch.triu(
            torch.ones(lookback, lookback, dtype=torch.bool, device=X.device),
            diagonal=1,
        )
        h = self.encoder(h, mask=causal_mask)  # [B*n_stocks, lookback, d_model]
        h_last = h[:, -1, :]                   # [B*n_stocks, d_model]
        return h_last.reshape(B, n_stocks, self.d_model)


class RelationMessagePassing(nn.Module):
    """Hand-rolled R-GCN layer: 3 relation-specific [in_dim, out_dim]
    weight matrices, aggregation via index_add_. No PyG dependency —
    with 40 nodes and 640 edges this is fast on CPU and GPU and carries
    zero dependency risk when Kaggle bumps its torch version.
    """

    def __init__(
        self,
        in_dim: int = 64,
        out_dim: int = 64,
        num_relations: int = 3,
    ) -> None:
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_relations = num_relations

        # One weight matrix per relation type (asserted at construction)
        self.W = nn.ParameterList([
            nn.Parameter(torch.empty(in_dim, out_dim))
            for _ in range(num_relations)
        ])
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.reset_parameters()
        assert len(self.W) == 3, (
            f"R-GCN must have exactly 3 relation weight matrices, got "
            f"{len(self.W)}"
        )

    def reset_parameters(self) -> None:
        for w in self.W:
            nn.init.xavier_uniform_(w)

    def forward(
        self,
        x: torch.Tensor,            # [N, in_dim]
        edge_index: torch.Tensor,    # [2, E]
        edge_type: torch.Tensor,     # [E]
    ) -> torch.Tensor:
        """Z = Σ_r ( aggregate_{j→i, r} x_j @ W_r ) + bias → [N, out_dim].

        Every node has the same degree per relation by graph
        construction (k_pos, k_neg out-edges per stock; 4 sector peers),
        so mean-vs-sum aggregation only differs by a constant factor
        per relation — absorbed by the weight matrices.
        """
        N = x.size(0)
        device = x.device
        out = torch.zeros(N, self.out_dim, device=device)

        for r in range(self.num_relations):
            mask = edge_type == r
            if not mask.any():
                continue
            src = edge_index[0, mask]
            dst = edge_index[1, mask]
            msgs = x[src] @ self.W[r]     # [E_r, out_dim]
            out = out.index_add(0, dst, msgs)

        return out + self.bias


def stack_per_sample_edges(
    edge_indices: Sequence[torch.Tensor],
    edge_types: Sequence[torch.Tensor],
    n_nodes: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Concatenate per-sample edge lists, shifting sample b's node ids by
    b*n_nodes.

    Flattening a batch of B dates gives [B*n_nodes, d_model] with sample
    b's nodes occupying rows [b*n_nodes, (b+1)*n_nodes). Shifting its node
    ids by that offset lets ONE index_add cover the whole batch instead of
    one R-GCN call per sample.

    The graph and the arithmetic are identical either way — every
    destination node accumulates the same source messages in the same
    order. What changes is the number of kernel launches: with 40 nodes
    and ~640 edges the per-sample call is launch-bound, not compute-bound,
    so B sequential calls cost roughly B times one batched call.
    """
    if len(edge_indices) != len(edge_types):
        raise ValueError(
            f"{len(edge_indices)} edge_index tensors but "
            f"{len(edge_types)} edge_type tensors"
        )
    if not edge_indices:
        raise ValueError("no per-sample edge lists given")
    ei = torch.cat([e + b * n_nodes for b, e in enumerate(edge_indices)],
                   dim=1)
    et = torch.cat(list(edge_types), dim=0)
    return ei, et


# ─── Mamba encoder (new in Phase 4; pure PyTorch, no compiled kernels) ─────

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def selective_scan(u, delta, A, B, C, D):
    # u, delta: (Bt, L, Di); A: (Di, S); B, C: (Bt, L, S); D: (Di,)
    Bt, L, Di = u.shape
    h = torch.zeros(Bt, Di, A.shape[1], device=u.device, dtype=u.dtype)
    ys = []
    for t in range(L):
        dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
        dBu = delta[:, t].unsqueeze(-1) * B[:, t].unsqueeze(1) * u[:, t].unsqueeze(-1)
        h = dA * h + dBu
        ys.append((h * C[:, t].unsqueeze(1)).sum(-1))
    return torch.stack(ys, dim=1) + u * D


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_inner, self.d_state = expand * d_model, d_state
        self.dt_rank = math.ceil(d_model / 16)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                groups=self.d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.last_delta = None                    # read-only, for Phase 4B signal + explainability

    def forward(self, x):
        L = x.shape[1]
        u, z = self.in_proj(x).chunk(2, dim=-1)
        u = F.silu(self.conv1d(u.transpose(1, 2))[..., :L].transpose(1, 2))   # causal conv
        dt, B, C = torch.split(self.x_proj(u), [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(dt))
        self.last_delta = delta.detach()
        y = selective_scan(u, delta, -torch.exp(self.A_log), B, C, self.D)
        return self.out_proj(y * F.silu(z))


class MambaEncoder(nn.Module):
    def __init__(self, n_features=8, d_model=64, n_layers=2, d_state=16, d_conv=4,
                 expand=2, dropout=0.1, d_out=None):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.layers = nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)])
        self.norms = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.final_norm = RMSNorm(d_model)
        self.out = nn.Linear(d_model, d_out) if (d_out and d_out != d_model) else nn.Identity()
        self.return_sequence = False              # True only inside the causality test

    def forward(self, x):
        h = self.input_proj(x)
        for norm, block in zip(self.norms, self.layers):
            h = h + self.dropout(block(norm(h)))
        h = self.final_norm(h)
        return h if self.return_sequence else self.out(h[:, -1])

    def delta_signal(self):
        """Mean memory-reset strength at the final step (last layer), one number per call."""
        return float(self.layers[-1].last_delta[:, -1].mean())

    def delta_per_sample(self, n_groups: int):
        """Per-group version of delta_signal for a batch of n_groups days:
        last layer, final step, averaged over stocks and channels → [n_groups]."""
        d = self.layers[-1].last_delta[:, -1]                # [n_groups * n_stocks, d_inner]
        return d.reshape(n_groups, -1).mean(-1)


class MambaStockEncoder(nn.Module):
    """Applies MambaEncoder independently to every stock's lookback window.
    X: [B, n_stocks, L, F] → [B, n_stocks, d_model] (same contract as
    TransformerEncoder.encode)."""

    def __init__(self, n_features=8, d_model=64, n_layers=2, d_state=16, d_conv=4,
                 expand=2, dropout=0.1):
        super().__init__()
        self.n_features, self.d_model = n_features, d_model
        self.mamba = MambaEncoder(n_features, d_model, n_layers, d_state, d_conv,
                                  expand, dropout, d_out=d_model)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        if X.dim() != 4:
            raise ValueError(f"encode() expects 4D input [B, n_stocks, L, F], got {tuple(X.shape)}")
        B, n_stocks, lookback, n_features = X.shape
        assert n_features == self.n_features
        h = self.mamba(X.reshape(B * n_stocks, lookback, n_features))   # [B*n_stocks, d_model]
        return h.reshape(B, n_stocks, self.d_model)

    def delta_signal(self):
        return self.mamba.delta_signal()

    def delta_per_day(self, n_days: int):
        return self.mamba.delta_per_sample(n_days)


# ─── DMRGT with a split forward ────────────────────────────────────────────

class DMRGT(nn.Module):
    """Same R-GCN + concat head as Phase 3; the temporal encoder is chosen
    by `encoder` ('transformer' → arm A, 'mamba' → arms B/C/D).

    Construction order (encoder → rgcn → head) is the Phase 3 order, so the
    transformer arm draws the same initial weights as Phase 3 for a seed.

      encode(x)               x [B, N, L, F]  → h [B, N, d]
      propagate(h, ei, et)    R-GCN + concat head → [B, N]
      forward                 propagate(encode(x), ...)
    """

    def __init__(self, encoder: str = "transformer", n_features: int = 8,
                 d_model: int = 64, n_heads: int = 4, n_layers: int = 2,
                 dim_ff: int = 256, dropout: float = 0.1, max_len: int = 64,
                 num_relations: int = 3, mamba_cfg: dict | None = None) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_model = d_model
        self.encoder_type = encoder
        if encoder == "transformer":
            self.encoder = TransformerEncoder(
                n_features=n_features, d_model=d_model, n_heads=n_heads,
                n_layers=n_layers, dim_ff=dim_ff, dropout=dropout, max_len=max_len)
        elif encoder == "mamba":
            mc = dict(mamba_cfg or {})
            self.encoder = MambaStockEncoder(
                n_features=n_features, d_model=mc.get("d_model", d_model),
                n_layers=mc.get("n_layers", 2), d_state=mc.get("d_state", 16),
                d_conv=mc.get("d_conv", 4), expand=mc.get("expand", 2),
                dropout=mc.get("dropout", dropout))
            assert mc.get("d_model", d_model) == d_model
        else:
            raise ValueError(f"unknown encoder {encoder!r}")
        self.rgcn = RelationMessagePassing(in_dim=d_model, out_dim=d_model,
                                           num_relations=num_relations)
        self.head = nn.Sequential(
            nn.Linear(2 * d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def _as_batched(self, X: torch.Tensor) -> tuple[torch.Tensor, bool]:
        if X.dim() == 3:
            x, squeeze = X.unsqueeze(0), True
        elif X.dim() == 4:
            x, squeeze = X, False
        else:
            raise ValueError(f"Expected X [n_stocks, L, F] or [B, n_stocks, L, F], got {tuple(X.shape)}")
        assert X.shape[-1] == self.n_features
        return x, squeeze

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        """X [N, L, F] or [B, N, L, F] → h [N, d] or [B, N, d]."""
        x, squeeze = self._as_batched(X)
        h = self.encoder.encode(x)
        return h.squeeze(0) if squeeze else h

    def propagate(self, h: torch.Tensor, edge_index: torch.Tensor,
                  edge_type: torch.Tensor) -> torch.Tensor:
        """h [N, d] or [B, N, d]; edges over the flattened batch nodes
        (stack_per_sample_edges). R-GCN + concat head → [N] or [B, N]."""
        squeeze = h.dim() == 2
        if squeeze:
            h = h.unsqueeze(0)
        B, n_stocks = h.size(0), h.size(1)
        Z = self.rgcn(h.reshape(B * n_stocks, self.d_model), edge_index,
                      edge_type).reshape(B, n_stocks, self.d_model)
        fused = torch.cat([h, Z], dim=-1)
        out = self.head(fused).squeeze(-1)
        return out.squeeze(0) if squeeze else out

    def propagate_with_edge_lists(self, h: torch.Tensor,
                                  edge_indices: Sequence[torch.Tensor],
                                  edge_types: Sequence[torch.Tensor]) -> torch.Tensor:
        B, n_stocks = h.size(0), h.size(1)
        if len(edge_indices) != B:
            raise ValueError(f"{len(edge_indices)} edge lists for a batch of {B}")
        ei, et = stack_per_sample_edges(edge_indices, edge_types, n_stocks)
        return self.propagate(h, ei, et)

    def forward(self, X: torch.Tensor, edge_index: torch.Tensor,
                edge_type: torch.Tensor) -> torch.Tensor:
        """One edge list applied to every sample in the batch."""
        x, squeeze = self._as_batched(X)
        B, n_stocks = x.size(0), x.size(1)
        if B == 1:
            ei, et = edge_index, edge_type
        else:
            ei, et = stack_per_sample_edges([edge_index] * B, [edge_type] * B, n_stocks)
        out = self.propagate(self.encoder.encode(x), ei, et)
        return out.squeeze(0) if squeeze else out

    def forward_with_edge_lists(self, X: torch.Tensor,
                                edge_indices: Sequence[torch.Tensor],
                                edge_types: Sequence[torch.Tensor]) -> torch.Tensor:
        """One edge list PER sample (a batch of dates). [B, n_stocks]."""
        x, squeeze = self._as_batched(X)
        out = self.propagate_with_edge_lists(self.encoder.encode(x), edge_indices, edge_types)
        return out.squeeze(0) if squeeze else out


def make_model(encoder: str, mamba_cfg: dict | None = None, n_features: int = 8) -> DMRGT:
    """The one constructor used by training, the probes and Phase 4B inference."""
    return DMRGT(encoder=encoder, n_features=n_features, mamba_cfg=mamba_cfg)


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_eval.py
"""DMRGT Phase 4 — evaluation harness (spec §6.4).

Ported verbatim from Phase 3 eval.py (incl. the source= keyword at both
load_predictions call sites). Phase 4 edits, all mechanical:
  * `import paths` → d4_data (cache / working-dir resolution);
  * evaluate_fold (the closed-form-baseline test path) asserts the test
    lock files exist before reading any test window.
The leg-direction correction and all metric formulas are unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Iterable

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import d4_data


# ─── per-day metrics (formulas unchanged from Phase 1/2) ─────────────────

def _huber_loss(residuals: np.ndarray, delta: float) -> np.ndarray:
    """Per-element Huber loss: quadratic for |r|<=delta, linear otherwise."""
    r = np.asarray(residuals, dtype=np.float64)
    abs_r = np.abs(r)
    quadratic = 0.5 * r * r
    linear = delta * (abs_r - 0.5 * delta)
    return np.where(abs_r <= delta, quadratic, linear)


def per_day_rank_ic(pred: np.ndarray, true: np.ndarray) -> float:
    """Spearman rank IC for a single cross-section. NaN if either side
    has zero variance; the aggregator drops NaN days."""
    if pred.size < 2:
        return float("nan")
    if np.std(pred) == 0 or np.std(true) == 0:
        return float("nan")
    rho, _ = spearmanr(pred, true)
    return float(rho) if np.isfinite(rho) else float("nan")


def per_day_directional_acc(pred: np.ndarray, true: np.ndarray) -> float:
    """Fraction of stocks with sign(pred) == sign(true), over valid pairs."""
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean(np.sign(pred[mask]) == np.sign(true[mask])))


def per_day_huber(pred: np.ndarray, true: np.ndarray, delta: float) -> float:
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean(_huber_loss(pred[mask] - true[mask], delta)))


def per_day_mse(pred: np.ndarray, true: np.ndarray) -> float:
    mask = np.isfinite(pred) & np.isfinite(true)
    if not mask.any():
        return float("nan")
    return float(np.mean((pred[mask] - true[mask]) ** 2))


# ─── dataclasses ──────────────────────────────────────────────────────────

@dataclass
class FoldMetrics:
    """All metrics for one model × fold × seed × split."""
    model: str
    fold: str
    seed: int | None
    split: str
    n_days: int
    # Prediction metrics — aggregated across the split's days
    mean_rank_ic: float
    std_rank_ic: float
    directional_acc: float
    huber_loss: float
    mse: float
    # Backtest metrics
    sharpe_after_costs: float
    max_drawdown: float
    turnover: float
    cum_return: float


@dataclass
class FoldResult:
    """Full output of evaluating one model on one fold.

    ``per_day`` is the un-aggregated per-day metric table;
    ``backtest_returns`` is the daily portfolio return series after
    costs (used by stats.py's bootstrap).
    """
    model: str
    fold: str
    seed: int | None
    split: str
    per_day: pd.DataFrame
    fold_metrics: FoldMetrics
    backtest_returns: pd.Series


def summarise(results: Iterable[FoldResult]) -> pd.DataFrame:
    """Flatten FoldResults into a tidy DataFrame."""
    return pd.DataFrame([asdict(r.fold_metrics) for r in results])


# ─── realised raw returns ─────────────────────────────────────────────────

def raw_returns_from_targets(targets: pd.DataFrame,
                             universe: list[str]) -> pd.DataFrame:
    """Recover realised RAW daily returns from the excess-return target.

    The backtest earns the raw return, not the demeaned target; raw[t] =
    excess[t] + cross-sectional mean[t]. Wide frame, date × universe.
    """
    tgt_wide = (targets.pivot(index="date", columns="ticker",
                              values="excess_return")
                .sort_index()
                .reindex(columns=universe))
    return tgt_wide.add(tgt_wide.mean(axis=1), axis=0)


# ─── post-processing (§9, applied identically to both arms — Rule 19) ─────

DEFAULT_POSTPROCESSING: dict = {
    "no_trade_band": 0.0,
    "decile_fraction": None,   # None → fall back to cfg backtest.decile_fraction
    "smoothing_days": 1,
}


def apply_postprocessing(preds: pd.DataFrame,
                         postprocessing: dict | None) -> pd.DataFrame:
    """Apply the Tier-1 post-processing rules to a long predictions frame.

    Rules, in order:
      1. smoothing_days — causal (trailing) rolling mean of y_pred over
         `smoothing_days` dates, per ticker, within each (model, fold,
         seed) group. Never looks ahead. 1 → no-op.
      2. no_trade_band — stocks whose |smoothed prediction| < band are
         flagged untradable on that date (``tradable`` column, False).
         The backtest excludes them from leg selection; rank IC keeps
         scoring them (the band is a trading rule, not a model change).

    decile_fraction travels in the same dict but is a backtest parameter
    (leg size), not a per-row transformation — backtest() reads it.

    Returns a new DataFrame; the input is not mutated.
    """
    if postprocessing is None:
        pp = dict(DEFAULT_POSTPROCESSING)
    else:
        pp = {**DEFAULT_POSTPROCESSING, **{
            k: v for k, v in postprocessing.items()
            if k in DEFAULT_POSTPROCESSING and v is not None
        }}

    df = preds.copy()
    if "tradable" not in df.columns:
        df["tradable"] = True

    # ── 1. causal smoothing ──────────────────────────────────────────
    smooth = int(pp.get("smoothing_days", 1) or 1)
    if smooth > 1:
        # Rolling per ticker within each (model, fold, seed) run — rows
        # of different tickers must never be mixed into one window.
        keys = [c for c in ("model", "fold", "seed", "ticker")
                if c in df.columns]
        df = df.sort_values(keys + ["date"])
        df["y_pred"] = (df.groupby(keys, sort=False)["y_pred"]
                        .transform(lambda s: s.rolling(smooth,
                                                       min_periods=1).mean()))
        df = df.reset_index(drop=True)

    # ── 2. no-trade band ─────────────────────────────────────────────
    band = float(pp.get("no_trade_band", 0.0) or 0.0)
    df["tradable"] = df["y_pred"].abs() >= band

    return df


# ─── the cost-aware backtest (§5.2) ───────────────────────────────────────

def backtest_daily_deciles(
    predictions_by_day: pd.DataFrame,   # date × ticker, values = y_pred
    true_returns_by_day: pd.DataFrame,  # date × ticker, values = realised raw return
    tickers: list[str],
    commission_bps: float,
    slippage_bps: float,
    decile_fraction: float,
    tradable_by_day: pd.DataFrame | None = None,  # date × ticker bool; None = all tradable
) -> tuple[pd.Series, dict]:
    """Long the top decile, short the bottom decile, equal-weighted within
    each leg. Legs are selected among the day's tradable stocks only.

    Positions decided from day-t predictions earn day-t+1 returns
    (weights are shifted one day before multiplying realised returns —
    no same-day lookahead). Commission + slippage are charged on
    weight changes: fee = (commission_bps + slippage_bps) / 1e4, applied
    to half the absolute daily weight change (one-way turnover).

    Returns (daily portfolio return series after costs, stats dict).
    """
    pred = predictions_by_day[tickers]
    rets = true_returns_by_day[tickers]
    trad = tradable_by_day[tickers] if tradable_by_day is not None else None

    n = len(tickers)
    fee = (commission_bps + slippage_bps) / 10_000.0

    weights = pd.DataFrame(0.0, index=pred.index, columns=tickers)
    for i, d in enumerate(pred.index):
        row = pred.loc[d].to_numpy(dtype=float)
        ok = (trad.loc[d].to_numpy(dtype=bool)
              if trad is not None else np.ones(n, dtype=bool))
        idx = np.flatnonzero(ok & np.isfinite(row))
        if idx.size == 0:
            continue  # nothing tradable → flat that day
        scores = row[idx]
        k = max(1, int(round(idx.size * decile_fraction)))
        k = min(k, idx.size)  # degenerate wide legs cancel to ~0 net
        order = np.argsort(-scores, kind="stable")  # descending: best first
        longs = idx[order[:k]]
        shorts = idx[order[-k:]]
        w = np.zeros(n)
        w[longs] += 1.0 / len(longs)
        w[shorts] -= 1.0 / len(shorts)
        weights.iloc[i] = w

    # Positions taken at close t earn returns realised at t+1.
    raw_port = (weights.shift(1).fillna(0) * rets).sum(axis=1)
    # Cost: one-way fee on each unit of weight change.
    w_change = weights.diff().abs().sum(axis=1).fillna(0)
    cost = fee * w_change / 2
    daily_ret = (raw_port - cost).astype(float)
    daily_ret.index = pd.DatetimeIndex(daily_ret.index)

    # Sharpe — annualised, risk-free 0, sample std.
    sd = float(daily_ret.std(ddof=1))
    sharpe = float(daily_ret.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0
    # Max drawdown from the equity curve.
    eq = (1 + daily_ret.fillna(0)).cumprod()
    peak = eq.cummax()
    dd = (eq - peak) / peak
    max_dd = float(-dd.min()) if dd.notna().any() else float("nan")
    # Turnover — one-way, fraction of NAV changing per day.
    turnover = float(w_change.mean() / 2)
    cum = float(eq.iloc[-1] - 1) if len(eq) > 0 else float("nan")

    return daily_ret, {
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "turnover": turnover,
        "cum_return": cum,
    }


def _effective_decile_fraction(cfg: dict, postprocessing: dict | None) -> float:
    """Post-processing decile_fraction overrides the config default."""
    if postprocessing and postprocessing.get("decile_fraction") is not None:
        return float(postprocessing["decile_fraction"])
    return float(cfg["backtest"]["decile_fraction"])


def _score_wide_group(
    wide_pred: pd.DataFrame,
    wide_true: pd.DataFrame,
    raw_wide: pd.DataFrame,
    tradable_wide: pd.DataFrame | None,
    cfg: dict,
    decile_fraction: float,
    universe: list[str],
) -> tuple[pd.DataFrame, pd.Series, dict]:
    """Per-day metrics + backtest for one (model, fold, seed) wide group."""
    dates = wide_pred.index
    per_day_rows = []
    for d in dates:
        y_pred = wide_pred.loc[d, universe].to_numpy()
        y_true = wide_true.loc[d, universe].to_numpy()
        per_day_rows.append({
            "date": d,
            "rank_ic": per_day_rank_ic(y_pred, y_true),
            "dir_acc": per_day_directional_acc(y_pred, y_true),
            "huber": per_day_huber(y_pred, y_true,
                                   delta=cfg["training"]["huber_delta"]),
            "mse": per_day_mse(y_pred, y_true),
        })
    per_day = pd.DataFrame(per_day_rows).set_index("date")

    bt_raw = raw_wide.reindex(dates).reindex(columns=universe)
    daily_ret, bt_stats = backtest_daily_deciles(
        predictions_by_day=wide_pred,
        true_returns_by_day=bt_raw,
        tickers=universe,
        commission_bps=cfg["backtest"]["commission_bps"],
        slippage_bps=cfg["backtest"]["slippage_bps"],
        decile_fraction=decile_fraction,
        tradable_by_day=tradable_wide,
    )
    return per_day, daily_ret, bt_stats


# ─── baseline path: model_fn over a fold's split (§5.3 gate) ──────────────

def evaluate_fold(
    model_name: str,
    model_fn: Callable[[np.ndarray], np.ndarray],
    fold,
    features: pd.DataFrame,
    targets: pd.DataFrame,
    cfg: dict,
    seed: int | None = None,
    lookback: int | None = None,
) -> FoldResult:
    """Run one closed-form model through the per-day harness on one fold.

    Assembles the same windows the trained model sees: for prediction
    date d, features from [d-lookback+1 .. d] INCLUSIVE of d (Rule 3 —
    features come from day t and earlier). Note: Phase 2's baseline
    harness used a window ending at d-1; the trained models' assembly
    always ended at d. Phase 3 unifies on the spec-correct inclusive
    window, so persistence numbers shift slightly vs Phase 2's recorded
    floor; the random gate is unaffected (noise ignores input).
    """
    d4_data.assert_test_unlocked()   # Phase 4: this path reads test windows
    universe = list(cfg["universe"]["tickers"])
    n_stocks = len(universe)
    if lookback is None:
        lookback = int(cfg["features"]["lookback_days"])
    feature_cols = list(cfg["features"]["columns"])

    # Reindex to guarantee the feature level follows configs.yaml order —
    # pivot's MultiIndex columns are sorted lexicographically, which would
    # silently reorder X's feature axis if configs ever listed features
    # out of alphabetical order.
    feat_wide = (features.pivot(index="date", columns="ticker",
                                values=feature_cols)
                 .sort_index())
    feat_wide = feat_wide.reindex(columns=pd.MultiIndex.from_product(
        [feature_cols, list(feat_wide.columns.get_level_values("ticker")
                            .unique())],
        names=["feature", "ticker"]))
    tgt_wide = (targets.pivot(index="date", columns="ticker",
                              values="excess_return")
                .sort_index())
    raw_wide = raw_returns_from_targets(targets, universe)

    test_dates = list(fold.test_dates)
    pred_rows, true_rows, daily_metrics = [], [], []
    for d in test_dates:
        if d not in feat_wide.index or d not in tgt_wide.index:
            continue  # e.g. last date dropped from targets
        d_loc = feat_wide.index.get_loc(d)
        if d_loc < lookback - 1:
            continue  # not enough history for an inclusive window
        window_dates = feat_wide.index[d_loc - lookback + 1: d_loc + 1]
        assert len(window_dates) == lookback
        assert window_dates[-1] == d
        X = np.empty((n_stocks, lookback, len(feature_cols)), dtype=np.float64)
        for j, t in enumerate(universe):
            X[j] = feat_wide.loc[window_dates, (slice(None), t)].values
        if not np.isfinite(X).all():
            continue
        y_pred = model_fn(X)
        y_true = tgt_wide.loc[d, universe].values
        pred_rows.append(pd.Series(y_pred, index=universe, name=d))
        true_rows.append(pd.Series(y_true, index=universe, name=d))
        daily_metrics.append({
            "date": d,
            "rank_ic": per_day_rank_ic(y_pred, y_true),
            "dir_acc": per_day_directional_acc(y_pred, y_true),
            "huber": per_day_huber(y_pred, y_true,
                                   delta=cfg["training"]["huber_delta"]),
            "mse": per_day_mse(y_pred, y_true),
        })

    if not daily_metrics:
        raise RuntimeError(
            f"No valid test days for fold {fold.name!r} — check that "
            f"features/targets cover the test window."
        )
    daily_df = pd.DataFrame(daily_metrics).set_index("date")
    pred_df = pd.DataFrame(pred_rows)
    true_df = pd.DataFrame(true_rows)

    _, daily_ret, bt_stats = _score_wide_group(
        pred_df, true_df, raw_wide, None, cfg,
        decile_fraction=float(cfg["backtest"]["decile_fraction"]),
        universe=universe,
    )

    metrics = FoldMetrics(
        model=model_name,
        fold=fold.name,
        seed=seed,
        split="test",
        n_days=len(daily_df),
        mean_rank_ic=float(np.nanmean(daily_df["rank_ic"])),
        std_rank_ic=float(np.nanstd(daily_df["rank_ic"], ddof=1)),
        directional_acc=float(np.nanmean(daily_df["dir_acc"])),
        huber_loss=float(np.nanmean(daily_df["huber"])),
        mse=float(np.nanmean(daily_df["mse"])),
        sharpe_after_costs=bt_stats["sharpe"],
        max_drawdown=bt_stats["max_drawdown"],
        turnover=bt_stats["turnover"],
        cum_return=bt_stats["cum_return"],
    )
    return FoldResult(
        model=model_name,
        fold=fold.name,
        seed=seed,
        split="test",
        per_day=daily_df,
        fold_metrics=metrics,
        backtest_returns=daily_ret,
    )


# ─── predictions-parquet path (trained models) ────────────────────────────

def load_predictions(model_id: str | None = None,
                     split: str | None = None,
                     source: str | Path | None = None) -> pd.DataFrame:
    """Load a Phase 3 predictions parquet.

    Resolution order: explicit `source` (file or directory) →
    WORK/predictions_{model_id}.parquet. Optionally filters to one
    `split` ("val" / "test").
    """
    src = Path(source) if source is not None else d4_data.WORK
    if src.is_dir():
        if model_id is None:
            matches = sorted(src.glob("predictions_*.parquet"))
            if not matches:
                raise FileNotFoundError(f"No predictions_*.parquet in {src}")
            if len(matches) > 1 and model_id is None:
                raise ValueError(
                    f"Multiple prediction files in {src}; pass model_id "
                    f"to disambiguate: {[m.name for m in matches]}"
                )
            src = matches[0]
        else:
            src = src / f"predictions_{model_id}.parquet"
    if not src.exists():
        raise FileNotFoundError(
            f"Predictions not found: {src}. Download the Kaggle output "
            f"into artifacts/ first."
        )
    df = pd.read_parquet(src)
    required = {"model", "fold", "seed", "split", "date", "ticker",
                "y_pred", "y_true"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"predictions parquet {src.name} missing columns: {missing}. "
            f"Have: {list(df.columns)}"
        )
    if split is not None:
        df = df[df["split"] == split]
        if df.empty:
            raise ValueError(
                f"{src.name} has no rows with split == {split!r}. "
                f"Splits present: {sorted(pd.read_parquet(src)['split'].unique())}"
            )
    return df


def evaluate_predictions(
    source: str | Path | pd.DataFrame,
    targets: pd.DataFrame,
    cfg: dict,
    split: str | None = None,
    postprocessing: dict | None = None,
    universe: list[str] | None = None,
) -> list[FoldResult]:
    """Re-evaluate a predictions parquet (Phase 3 schema with a `split`
    column) through the same code path as evaluate_fold. One FoldResult
    per (model, fold, seed) — the pairing stats.py's Wilcoxon test uses.
    """
    if universe is None:
        universe = list(cfg["universe"]["tickers"])
    pred_df = (source if isinstance(source, pd.DataFrame)
               else load_predictions(source=source, split=split))
    pred_df = apply_postprocessing(pred_df, postprocessing)

    if split is not None:
        pred_df = pred_df[pred_df["split"] == split]
    if pred_df.empty:
        raise ValueError(f"No prediction rows to evaluate (split={split!r})")
    if pred_df["model"].nunique() > 1:
        raise ValueError(
            "evaluate_predictions refuses to pool multiple models into one "
            "result set — evaluate one arm at a time so the two arms of "
            "the comparison can never be silently mixed."
        )

    raw_wide = raw_returns_from_targets(targets, universe)
    frac = _effective_decile_fraction(cfg, postprocessing)

    results: list[FoldResult] = []
    for (model_name, fold_name, seed), grp in pred_df.groupby(
            ["model", "fold", "seed"]):
        # A group that mixed splits would silently blend the val and test
        # windows into one return series — refuse loudly instead.
        assert grp["split"].nunique() == 1, (
            f"group model={model_name} fold={fold_name} seed={seed} spans "
            f"splits {sorted(grp['split'].unique())} — filter with "
            f"split= before evaluating; never pool the tuning and scoring "
            f"windows."
        )
        wide_pred = (grp.pivot(index="date", columns="ticker",
                               values="y_pred").sort_index()
                     .reindex(columns=universe))
        wide_true = (grp.pivot(index="date", columns="ticker",
                               values="y_true").sort_index()
                     .reindex(columns=universe))
        tradable_wide = None
        if "tradable" in grp.columns:
            tradable_wide = (grp.pivot(index="date", columns="ticker",
                                       values="tradable").sort_index()
                             .reindex(columns=universe))

        per_day, daily_ret, bt_stats = _score_wide_group(
            wide_pred, wide_true, raw_wide, tradable_wide, cfg,
            decile_fraction=frac, universe=universe,
        )
        split_val = str(grp["split"].iloc[0])
        metrics = FoldMetrics(
            model=str(model_name),
            fold=str(fold_name),
            seed=int(seed),
            split=split_val,
            n_days=len(per_day),
            mean_rank_ic=float(np.nanmean(per_day["rank_ic"])),
            std_rank_ic=float(np.nanstd(per_day["rank_ic"], ddof=1)),
            directional_acc=float(np.nanmean(per_day["dir_acc"])),
            huber_loss=float(np.nanmean(per_day["huber"])),
            mse=float(np.nanmean(per_day["mse"])),
            sharpe_after_costs=bt_stats["sharpe"],
            max_drawdown=bt_stats["max_drawdown"],
            turnover=bt_stats["turnover"],
            cum_return=bt_stats["cum_return"],
        )
        results.append(FoldResult(
            model=str(model_name),
            fold=str(fold_name),
            seed=int(seed),
            split=split_val,
            per_day=per_day,
            fold_metrics=metrics,
            backtest_returns=daily_ret,
        ))
    return results


# ─── aggregate backtest — the grid's / compare.py interface ───────────────

def backtest(
    predictions: pd.DataFrame | str | Path,
    cfg: dict,
    split: str | None = None,
    postprocessing: dict | None = None,
    targets: pd.DataFrame | None = None,
) -> dict:
    """Score a predictions frame through the cost-aware backtest and
    aggregate across every (model, fold, seed) group it contains.

    Aggregation is the MEAN across groups (4 folds × 5 seeds = 20 runs
    per arm): mean of per-run mean rank ICs, mean of per-run Sharpes,
    mean of per-run max drawdowns and turnovers. Returns one flat dict;
    `sharpe` is the key the grid's selection_metric reads.

    Refuses to pool more than one model — a pooled aggregate across the
    two arms would be meaningless.
    """
    df = (predictions if isinstance(predictions, pd.DataFrame)
          else load_predictions(source=predictions, split=split))
    if targets is None:
        targets = pd.read_parquet(d4_data.resolve_cache() / "targets.parquet")
    results = evaluate_predictions(df, targets, cfg, split=split,
                                   postprocessing=postprocessing)
    if not results:
        raise ValueError("No (model, fold, seed) groups to backtest")

    per_run = []
    for r in results:
        m = asdict(r.fold_metrics)
        m["sharpe"] = m["sharpe_after_costs"]
        per_run.append(m)
    keys = ["mean_rank_ic", "std_rank_ic", "directional_acc", "huber_loss",
            "mse", "sharpe_after_costs", "max_drawdown", "turnover",
            "cum_return"]
    agg = {k: float(np.mean([run[k] for run in per_run])) for k in keys}
    agg["sharpe"] = agg["sharpe_after_costs"]
    agg["n_runs"] = len(per_run)
    agg["n_days"] = int(sum(run["n_days"] for run in per_run))
    agg["per_run"] = per_run
    agg["split"] = split
    return agg

# ─── closed-form baselines (ported verbatim from baselines.py) ────────────

def random_model(X: np.ndarray, seed: int | None = None) -> np.ndarray:
    """Ignore input, return standard-normal noise of shape [n_stocks]."""
    rng = np.random.default_rng(seed)
    return rng.standard_normal(X.shape[0])


def persistence_model(X: np.ndarray, feature_idx_logret: int = 0) -> np.ndarray:
    """Predict tomorrow's excess return = today's log-return, demeaned.

    Uses X[:, -1, feature_idx_logret] only — the most recent day's
    log-return feature (the last row of the lookback window, i.e. the
    prediction date itself). The mean subtraction matches the target's
    cross-sectional demeaning.
    """
    today_return = X[:, -1, feature_idx_logret]
    return today_return - np.nanmean(today_return)

# ─── Phase 4 addition: prediction-frame helper (schema of load_predictions) ──

def make_pred_frame(model_name: str, fold: str, seed: int, split: str, dates,
                    tickers, preds: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    n, N = preds.shape
    assert y.shape == preds.shape and len(tickers) == N and len(dates) == n
    return pd.DataFrame({
        "model": model_name, "fold": fold, "seed": int(seed), "split": split,
        "date": np.repeat(pd.to_datetime(np.asarray(dates)).values, N),
        "ticker": np.tile(np.array(list(tickers)), n),
        "y_pred": np.asarray(preds, dtype=np.float64).reshape(-1),
        "y_true": np.asarray(y, dtype=np.float64).reshape(-1),
    })


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_stats.py
"""DMRGT Phase 4 — significance testing (spec §6.4).

Ported verbatim from Phase 3 stats.py: Wilcoxon signed-rank on paired
(fold, seed) rank IC, and the NumPy Politis–Romano stationary bootstrap on
the Sharpe difference (seed-averaged per fold, folds pooled, paired
resampling). Phase 4 edits: imports (paths/data → d4_data, eval → d4_eval),
the local CLI was dropped, and the Holm helper + generic wrappers at the
bottom were added. The "dynamic"/"frozen" key names inside the ported
functions are kept as-is: in Phase 4, "dynamic" = arm/policy A under test
and "frozen" = the reference it is compared to.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon as scipy_wilcoxon

import d4_data
import d4_eval as ev


# ─── inputs assembled FROM prediction frames ───────────────────────────────

def _as_frame(pred) -> pd.DataFrame:
    """Accept a predictions DataFrame or a path to its parquet."""
    if isinstance(pred, pd.DataFrame):
        df = pred
    else:
        df = pd.read_parquet(pred)
    return df


def _test_rows(pred) -> pd.DataFrame:
    """The test-split rows of a predictions file/frame.

    Callers run strictly after winner selection is locked (§1), so this
    is the first and only place test rows are read.
    """
    df = _as_frame(pred)
    assert df["model"].nunique() == 1, (
        "a significance-test input must hold exactly one arm's rows"
    )
    out = df[df["split"] == "test"]
    assert len(out) > 0, "no test rows in the predictions input"
    return out


def per_fold_seed_ic(pred_df: pd.DataFrame, cfg: dict) -> dict:
    """{(fold, seed): mean rank IC} for one arm's TEST rows — the 20
    paired observations the Wilcoxon test consumes.

    Scores through the same evaluation pipeline as everything else
    (identity post-processing, the declared final configuration), so the
    ICs here are exactly the per-run mean_rank_ic values compare.py
    reports.
    """
    targets = pd.read_parquet(d4_data.resolve_cache() / "targets.parquet")
    runs = ev.evaluate_predictions(_test_rows(pred_df), targets, cfg,
                                   split=None)
    keyed = {(r.fold, r.seed): float(r.fold_metrics.mean_rank_ic)
             for r in runs}
    assert len(keyed) == len(runs), "duplicate (fold, seed) keys"
    return keyed


def daily_returns(pred_df: pd.DataFrame, cfg: dict) -> pd.Series:
    """Pooled daily after-cost return series for one arm's TEST rows.

    Seed-averaged per (model, fold), then folds concatenated
    chronologically (by each series' first date — disjoint calendar
    years make the stitch points unambiguous).
    """
    targets = pd.read_parquet(d4_data.resolve_cache() / "targets.parquet")
    runs = ev.evaluate_predictions(_test_rows(pred_df), targets, cfg,
                                   split=None)
    by_fold: dict[str, list[pd.Series]] = {}
    for r in runs:
        by_fold.setdefault(r.fold, []).append(r.backtest_returns)
    per_fold = []
    for fold, series in by_fold.items():
        df = pd.concat(series, axis=1)
        assert df.index.is_monotonic_increasing, f"{fold}: unordered index"
        per_fold.append(df.mean(axis=1))
    per_fold.sort(key=lambda s: s.index[0])
    return pd.concat(per_fold)


# ─── the two tests (§6) ────────────────────────────────────────────────────

def _sharpe(daily_ret: np.ndarray) -> float:
    """Annualised Sharpe of a daily return series (risk-free 0)."""
    r = np.asarray(daily_ret, dtype=float)
    if r.size < 2:
        return float("nan")
    sd = float(np.std(r, ddof=1))
    return float(np.mean(r) / sd * np.sqrt(252)) if sd > 0 else 0.0


def _sharpe_rows(returns: np.ndarray) -> np.ndarray:
    """Annualised Sharpe of each row of a [reps, n] return matrix."""
    sd = returns.std(axis=1, ddof=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        s = returns.mean(axis=1) / sd * np.sqrt(252)
    return np.where(sd > 0, s, 0.0)


def _stationary_bootstrap_indices(n: int, block: int, reps: int,
                                  seed: int) -> np.ndarray:
    """Politis–Romano stationary bootstrap index draws, shape [reps, n].

    Each replicate starts at a uniform index; every subsequent index
    follows the previous one (t+1 mod n — circular, as in the original
    scheme) with probability 1 − 1/block, else jumps to a fresh uniform
    index. Expected block length is therefore ``block``.
    """
    if not (1 <= block <= n):
        raise ValueError(f"block length {block} out of range for n={n}")
    rng = np.random.default_rng(seed)
    out = np.empty((reps, n), dtype=np.int64)
    cur = rng.integers(0, n, size=reps)
    out[:, 0] = cur
    p_follow = 1.0 - 1.0 / float(block)
    for t in range(1, n):
        follow = rng.random(reps) < p_follow
        cur = np.where(follow, (cur + 1) % n, rng.integers(0, n, size=reps))
        out[:, t] = cur
    return out


def _stationary_bootstrap_diffs(dyn: np.ndarray, fro: np.ndarray,
                                block: int, reps: int, seed: int) -> np.ndarray:
    """Paired stationary bootstrap on the Sharpe difference, [reps].

    One index draw per replicate, applied to BOTH arms — the replicate
    isolates the arms, not the resample. numpy instead of arch's
    StationaryBootstrap because Cell 6 runs in-session with Internet
    OFF and arch is not in Kaggle's base image; same Politis–Romano
    scheme (expected block length ``block``, circular indexing).
    """
    dyn = np.asarray(dyn, dtype=float)
    fro = np.asarray(fro, dtype=float)
    idx = _stationary_bootstrap_indices(len(dyn), block, reps, seed)
    return _sharpe_rows(dyn[idx]) - _sharpe_rows(fro[idx])


def wilcoxon_test(dynamic_ic, frozen_ic, cfg: dict) -> dict:
    """Wilcoxon signed-rank on paired (fold, seed) mean rank ICs.

    ``dynamic_ic`` / ``frozen_ic`` are the per_fold_seed_ic() dicts (or
    equal-length sequences, paired by sort order). The result carries
    ``wins`` — how many paired observations the dynamic arm won — which
    the §6 conclusion string reports.
    """
    if isinstance(dynamic_ic, dict):
        assert isinstance(frozen_ic, dict)
        keys = sorted(set(dynamic_ic) & set(frozen_ic))
        missing = set(dynamic_ic) ^ set(frozen_ic)
        assert not missing, (
            f"unpaired (fold, seed) runs between arms: {sorted(missing)} — "
            f"the Wilcoxon test needs every run in BOTH arms"
        )
        dyn = [dynamic_ic[k] for k in keys]
        fro = [frozen_ic[k] for k in keys]
    else:
        dyn, fro = list(dynamic_ic), list(frozen_ic)
    assert len(dyn) == len(fro), (
        f"unpaired samples: {len(dyn)} dynamic vs {len(fro)} frozen"
    )

    alpha = float(cfg["significance"]["wilcoxon_alpha"])
    stat, p = scipy_wilcoxon(dyn, fro)
    return {
        "test": "Wilcoxon signed-rank, paired on (fold, seed)",
        "n_pairs": len(dyn),
        "wins": int(sum(1 for a, b in zip(dyn, fro) if a > b)),
        "statistic": float(stat),
        "p_value": float(p),
        "alpha": alpha,
        "significant": bool(p < alpha),
        "dynamic_mean_rank_ic": float(np.mean(dyn)),
        "frozen_mean_rank_ic": float(np.mean(fro)),
        "mean_paired_difference": float(np.mean(
            [a - b for a, b in zip(dyn, fro)])),
    }


def bootstrap_sharpe_test(dynamic_returns: pd.Series,
                          frozen_returns: pd.Series,
                          cfg: dict, seed: int = 7) -> dict:
    """Stationary bootstrap on the Sharpe difference between arms.

    Paired resampling: each replicate draws one index sequence and
    applies it to BOTH arms; the replicate statistic is
    Sharpe(dyn_resample) − Sharpe(fro_resample).

    ``bootstrap_p_value`` is the ONE-SIDED p under the pre-declared H1
    (dynamic > frozen) — the number the §6 conclusion string reports.
    """
    sig = cfg["significance"]
    n_iterations = int(sig["bootstrap_iterations"])
    block = int(sig["bootstrap_block_length"])
    alpha = float(sig["wilcoxon_alpha"])

    dyn = np.asarray(dynamic_returns, dtype=float)
    fro = np.asarray(frozen_returns, dtype=float)
    assert len(dyn) == len(fro) > 0, "pooled return series misaligned"

    observed = _sharpe(dyn) - _sharpe(fro)

    # Paired resampling: each replicate draws ONE index sequence and
    # applies it to BOTH arms, so the replicate statistic isolates the
    # arms, not the resample.
    diffs = _stationary_bootstrap_diffs(dyn, fro, block, n_iterations, seed)
    assert diffs.shape == (n_iterations,)

    p_one = float(np.mean(diffs <= 0))          # H1: dynamic > frozen
    p_two = float(min(1.0, 2 * min(p_one, 1 - p_one)))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {
        "test": "Stationary bootstrap (Politis–Romano) on Sharpe difference",
        "n_iterations": n_iterations,
        "block_length": block,
        "construction": (
            "daily after-cost returns, seed-averaged per (model, fold), "
            "pooled across folds chronologically, paired resampling"
        ),
        "observed_sharpe_dynamic": _sharpe(dyn),
        "observed_sharpe_frozen": _sharpe(fro),
        "observed_sharpe_difference": float(observed),
        "ci_95": [float(lo), float(hi)],
        "p_one_sided_dynamic_greater": p_one,
        "p_two_sided": p_two,
        "bootstrap_p_value": p_one,
        "alpha": alpha,
        "significant_one_sided": bool(p_one < alpha),
        "bootstrap_mean_difference": float(np.mean(diffs)),
    }


# ─── the composite the notebook's Cell 6 and compare.py both call ──────────

def run_tests_from_predictions(dynamic_pred, frozen_pred, cfg: dict,
                               winning_graph_config: dict | None = None,
                               ) -> dict:
    """Both tests from the two arms' prediction files/frames, returning
    the dict phase3_significance_results.json holds (§6's exact shape:
    wilcoxon, bootstrap_sharpe, winning_graph_config, conclusion).

    The dynamic input is the WINNING grid variant's prediction file —
    its test rows are the final dmrgt_all3 result (§1: never retrained).
    """
    dyn = _test_rows(dynamic_pred)
    fro = _test_rows(frozen_pred)

    wilcoxon_result = wilcoxon_test(
        dynamic_ic=per_fold_seed_ic(dyn, cfg),
        frozen_ic=per_fold_seed_ic(fro, cfg),
        cfg=cfg,
    )
    bootstrap_result = bootstrap_sharpe_test(
        dynamic_returns=daily_returns(dyn, cfg),
        frozen_returns=daily_returns(fro, cfg),
        cfg=cfg,
    )

    conclusion = (
        f"Dynamic graph rank IC exceeds frozen in {wilcoxon_result['wins']}/"
        f"{wilcoxon_result['n_pairs']} folds (p={wilcoxon_result['p_value']:.3f}); "
        f"Sharpe difference p={bootstrap_result['bootstrap_p_value']:.3f}"
    )
    return {
        "wilcoxon": wilcoxon_result,
        "bootstrap_sharpe": bootstrap_result,
        "winning_graph_config": winning_graph_config,
        "conclusion": conclusion,
    }


# ─── Phase 4 additions ─────────────────────────────────────────────────────

def holm_correction(p_values: dict, alpha: float = 0.05) -> dict:
    """Holm–Bonferroni step-down. Returns {name: {p_raw, p_adj, reject}}."""
    items = sorted(p_values.items(), key=lambda kv: kv[1])
    m = len(items)
    out, running = {}, 0.0
    for rank, (name, p) in enumerate(items):
        adj = min(1.0, (m - rank) * p)
        running = max(running, adj)          # enforce monotonicity
        out[name] = {"p_raw": float(p), "p_adj": float(running), "reject": bool(running < alpha)}
    return out


def compare_arms(test_pred, ref_pred, cfg: dict) -> dict:
    """Wilcoxon + bootstrap for one comparison of two prediction frames
    (test rows). `test_pred` plays the 'dynamic' role, `ref_pred` the
    'frozen' role of the ported functions."""
    a, b = _test_rows(test_pred), _test_rows(ref_pred)
    wil = wilcoxon_test(per_fold_seed_ic(a, cfg), per_fold_seed_ic(b, cfg), cfg)
    boot = bootstrap_sharpe_test(daily_returns(a, cfg), daily_returns(b, cfg), cfg)
    return {"wilcoxon": wil, "bootstrap_sharpe": boot}


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_train.py
"""DMRGT Phase 4 — training (spec §6.5).

train_one(arm, fold, seed, batch_size): seeded CPU construction + byte
fingerprint (Phase 3 Rule 15 assertion ported), AdamW + linear warmup +
ReduceLROnPlateau on validation rank IC, gradient clipping, early stopping
on validation mean rank IC, optional adaptive Huber delta (arm D) from the
epoch's TRAINING residuals. Only the train and val splits are ever
requested — no test data is touched here.
"""

from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import d4_data
import d4_eval as ev
import d4_graph
import d4_models


# ─── seeding + init fingerprint (ported from Phase 3 train.py) ─────────────

def set_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def param_fingerprint(model: torch.nn.Module) -> str:
    """Byte-level fingerprint of a model's trainable parameters (device-stable)."""
    h = hashlib.sha256()
    with torch.no_grad():
        for p in model.parameters():
            if p.requires_grad:
                h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def arm_cfg(arm: str) -> dict:
    return d4_data.load_snapshot()["CONFIG"]["arms"][arm]


def build_model(arm: str, snapshot: dict | None = None):
    snapshot = snapshot or d4_data.load_snapshot()
    a = snapshot["CONFIG"]["arms"][arm]
    return d4_models.make_model(a["encoder"], snapshot["CONFIG"]["mamba"])


def ckpt_paths(out_dir: Path, arm: str, fold: str, seed: int) -> dict:
    tag = f"{arm}_{fold}_{seed}"
    return {"ckpt": out_dir / f"ckpt_{tag}.pt", "valpred": out_dir / f"valpred_{tag}.parquet",
            "log": out_dir / f"log_{tag}.csv", "done": out_dir / f"done_{tag}.json"}


def score_val(pred: np.ndarray, y: np.ndarray, dates, raw_wide: pd.DataFrame,
              universe: list, bt_cfg: dict) -> tuple[float, float]:
    """Validation (mean rank IC, Sharpe after costs) through the ported harness."""
    ics = [ev.per_day_rank_ic(pred[i], y[i]) for i in range(len(pred))]
    mean_ic = float(np.nanmean(ics)) if np.isfinite(ics).any() else float("nan")
    wide_pred = pd.DataFrame(pred, index=pd.DatetimeIndex(dates), columns=universe)
    bt_raw = raw_wide.reindex(wide_pred.index).reindex(columns=universe)
    _, st = ev.backtest_daily_deciles(
        wide_pred, bt_raw, universe, bt_cfg["commission_bps"], bt_cfg["slippage_bps"],
        float(bt_cfg["decile_fraction"]))
    return mean_ic, float(st["sharpe"])


def train_one(arm: str, fold: str, seed: int, batch_size: int, max_epochs: int | None = None,
              out_dir: Path | None = None, shuffle_targets: bool = False,
              write: bool = True) -> dict:
    snap = d4_data.load_snapshot()
    C = snap["CONFIG"]
    tc, a = C["training"], C["arms"][arm]
    out_dir = Path(out_dir) if out_dir is not None else d4_data.WORK
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

    t_start = time.time()
    lookback = int(a["lookback"])
    cfg = d4_data.load_cfg()
    universe = list(cfg["universe"]["tickers"])
    sd_tr = d4_graph.get_split_g(fold, "train", lookback)
    sd_va = d4_graph.get_split_g(fold, "val", lookback)
    edges = d4_graph.load_daily_edges()
    y_train = sd_tr.y
    if shuffle_targets:                       # leakage gate: same days, labels scrambled across dates
        y_train = y_train[np.random.default_rng(12345).permutation(len(y_train))]

    # ── Rule 15: random-init assertion, on CPU ────────────────────────
    set_seed(seed)
    net = build_model(arm, snap)
    fp1 = param_fingerprint(net)
    set_seed(seed)
    net2 = build_model(arm, snap)
    assert fp1 == param_fingerprint(net2), (
        f"Random-init assertion failed: same seed produced different params (seed={seed}).")
    set_seed(seed + 9999)
    net3 = build_model(arm, snap)
    assert fp1 != param_fingerprint(net3), (
        f"Random-init assertion failed: different seeds produced identical params (seed={seed}).")
    del net2, net3
    net = net.to(device)
    set_seed(seed)                            # restart the RNG at the seeded point
    n_params = int(sum(p.numel() for p in net.parameters()))

    X_tr = torch.from_numpy(sd_tr.X).float().to(device)
    y_tr = torch.from_numpy(y_train).float().to(device)
    X_va = torch.from_numpy(sd_va.X).float().to(device)
    tr_ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd_tr.dates]
    tr_et = [edges[pd.Timestamp(d)][1].to(device) for d in sd_tr.dates]
    va_ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd_va.dates]
    va_et = [edges[pd.Timestamp(d)][1].to(device) for d in sd_va.dates]
    raw_wide = ev.raw_returns_from_targets(d4_data.tables()["targets"], universe)

    opt = torch.optim.AdamW(net.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
    pl = tc["plateau"]
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode=pl["mode"], factor=pl["factor"], patience=pl["patience"], min_lr=pl["min_lr"])
    warm, lr0, lr1 = int(tc["warmup_epochs"]), tc["warmup_start_lr"], tc["lr"]
    epochs = int(max_epochs if max_epochs is not None else tc["max_epochs"])
    patience = int(tc["early_stop_patience"])
    adaptive = bool(a["adaptive_huber"])
    ah = tc["adaptive_huber"]
    huber_delta = float(tc["huber_delta_fixed"])

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log, best_ic, best_state, best_epoch, bad = [], -float("inf"), None, -1, 0
    n_train = len(X_tr)

    for epoch in range(epochs):
        t_ep = time.time()
        if epoch < warm:                      # linear warmup 1e-4 → 1e-3
            lr = lr0 + (lr1 - lr0) * epoch / max(1, warm - 1)
            for g in opt.param_groups:
                g["lr"] = lr
        lr_now = float(opt.param_groups[0]["lr"])

        net.train()
        perm = torch.randperm(n_train, device=device)
        ep_loss, n_b, resid = 0.0, 0, []
        for i in range(0, n_train, batch_size):
            idx = perm[i: i + batch_size]
            rows = idx.tolist()
            opt.zero_grad()
            yp = net.forward_with_edge_lists(X_tr[idx], [tr_ei[j] for j in rows], [tr_et[j] for j in rows])
            yb = y_tr[idx]
            loss = F.huber_loss(yp, yb, delta=huber_delta)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), tc["grad_clip_norm"])
            opt.step()
            ep_loss += float(loss.item())
            n_b += 1
            if adaptive:
                resid.append((yp.detach() - yb).reshape(-1))
        train_loss = ep_loss / max(1, n_b)

        net.eval()
        val_pred = predict(net, X_va, va_ei, va_et, batch_size)
        val_ic, val_sh = score_val(val_pred, sd_va.y, sd_va.dates, raw_wide, universe, C["backtest"])
        log.append({"epoch": epoch, "lr": lr_now, "huber_delta": huber_delta,
                    "train_loss": train_loss, "val_rank_ic": val_ic, "val_sharpe": val_sh,
                    "epoch_seconds": time.time() - t_ep})

        if adaptive:                          # next epoch's delta, from THIS epoch's training residuals
            r = torch.cat(resid)
            mad = torch.median(torch.abs(r - torch.median(r)))
            huber_delta = float(torch.clamp(ah["c"] * 1.4826 * mad, min=ah["min"], max=ah["max"]))

        score = val_ic if np.isfinite(val_ic) else -float("inf")
        if score > best_ic:
            best_ic, best_epoch, bad = score, epoch, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
        if epoch >= warm:                     # plateau scheduler is not stepped during warmup
            sched.step(score if np.isfinite(score) else -1.0)
        if bad >= patience:
            break

    if best_state is not None:
        net.load_state_dict(best_state)
    net.eval()
    val_pred = predict(net, X_va, va_ei, va_et, batch_size)
    assert val_pred.shape == (len(sd_va.dates), len(universe))
    peak = (torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0
    best_row = log[best_epoch] if best_epoch >= 0 else log[-1]
    result = {
        "arm": arm, "fold": fold, "seed": seed, "lookback": lookback, "batch_size": batch_size,
        "fingerprint": fp1, "n_params": n_params,
        "epochs_run": len(log), "best_epoch": best_epoch,
        "best_val_rank_ic": float(best_row["val_rank_ic"]), "best_val_sharpe": float(best_row["val_sharpe"]),
        "n_train_days": int(n_train), "n_val_days": int(len(sd_va.dates)),
        "mean_epoch_seconds": float(np.mean([r["epoch_seconds"] for r in log])),
        "last_epoch_seconds": float(log[-1]["epoch_seconds"]),
        "peak_gpu_gb": float(peak), "total_seconds": time.time() - t_start,
        "shuffled_targets": bool(shuffle_targets),
    }
    if write:
        p = ckpt_paths(out_dir, arm, fold, seed)
        torch.save({"arm": arm, "fold": fold, "seed": seed, "lookback": lookback,
                    "encoder": a["encoder"], "model_state_dict": best_state,
                    "scaler_mean": d4_data.get_builder(fold, lookback).scaler_mean,
                    "scaler_scale": d4_data.get_builder(fold, lookback).scaler_scale},
                   p["ckpt"])
        ev.make_pred_frame(arm, fold, seed, "val", sd_va.dates, universe, val_pred, sd_va.y) \
            .to_parquet(p["valpred"], index=False)
        pd.DataFrame(log).to_csv(p["log"], index=False)
        p["done"].write_text(json.dumps(result, indent=2))      # marker written last
    result["val_pred"] = val_pred
    return result


def predict(net, X, ei, et, batch_size: int) -> np.ndarray:
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            n_b = min(batch_size, len(X) - i)
            out.append(net.forward_with_edge_lists(X[i: i + n_b], ei[i: i + n_b], et[i: i + n_b])
                       .cpu().numpy())
    return np.concatenate(out, 0)


def probe_memory(arm: str, fold: str, batch_size: int) -> float:
    """One forward+backward at `batch_size`; returns peak GB (0 on CPU)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    a = arm_cfg(arm)
    sd = d4_graph.get_split_g(fold, "train", int(a["lookback"]))
    edges = d4_graph.load_daily_edges()
    set_seed(0)
    net = build_model(arm).to(device)
    n = min(batch_size, len(sd.X))
    X = torch.from_numpy(sd.X[:n]).float().to(device)
    y = torch.from_numpy(sd.y[:n]).float().to(device)
    ei = [edges[pd.Timestamp(d)][0].to(device) for d in sd.dates[:n]]
    et = [edges[pd.Timestamp(d)][1].to(device) for d in sd.dates[:n]]
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    net.train()
    loss = F.huber_loss(net.forward_with_edge_lists(X, ei, et), y)
    loss.backward()
    peak = (torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0
    del net, X, y, loss
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return float(peak)


In [ ]:
%%writefile /kaggle/working/dmrgt4/d4_worker.py
"""DMRGT Phase 4 — one worker process per GPU (spec §6.6).

usage: python d4_worker.py --jobs /kaggle/working/jobs_gpu0.json --batch-size 32
Processes, not threads: no shared RNG state. Errors never stop the other jobs.
"""

import argparse
import json
import os
import sys
import time
import traceback
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, str(Path(__file__).resolve().parent))

import d4_data
import d4_train


def done_marker(job):
    return d4_train.ckpt_paths(d4_data.WORK, job["arm"], job["fold"], job["seed"])["done"]


def error_file(job, attempt):
    return d4_data.WORK / f"error_{job['arm']}_{job['fold']}_{job['seed']}_attempt{attempt}.json"


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", required=True)
    ap.add_argument("--batch-size", type=int, required=True)
    args = ap.parse_args()
    jobs = json.loads(Path(args.jobs).read_text())
    for job in jobs:
        if done_marker(job).exists():
            continue                                  # safe if cell 13 is ever re-entered
        for attempt in (1, 2):                        # retry once on failure
            try:
                t0 = time.time()
                r = d4_train.train_one(**job, batch_size=args.batch_size)
                print(f"[worker] done {job} epochs={r['epochs_run']} "
                      f"ic={r['best_val_rank_ic']:+.4f} {time.time() - t0:.0f}s", flush=True)
                break
            except Exception:
                tb = traceback.format_exc()
                print(f"[worker] FAILED {job} attempt {attempt}\n{tb}", flush=True)
                write_json(error_file(job, attempt), {"job": job, "traceback": tb})


if __name__ == "__main__":
    main()


In [ ]:
# ─── Cell 9b: import the modules written above ───
import sys, importlib
sys.path.insert(0, "/kaggle/working/dmrgt4")
os.environ["DMRGT4_WORK"] = str(WORK)
import d4_data, d4_models, d4_graph, d4_eval, d4_stats, d4_train
for _m in (d4_data, d4_models, d4_graph, d4_eval, d4_stats, d4_train):
    importlib.reload(_m)
import numpy as np, pandas as pd
cfg = d4_data.load_cfg()
UNIVERSE = list(cfg["universe"]["tickers"])
print("modules imported; test split locked:", not d4_data.test_unlocked())


In [ ]:
# ─── Cell 10: unit tests (hard stop on any failure) ───
import hashlib
import torch
import d4_models, d4_graph, d4_data

_t0 = time.time()
UT = {}
_mc = CONFIG["mamba"]

# 1. Mamba causality — future days must not leak into past states
torch.manual_seed(0)
_enc = d4_models.MambaEncoder(8, _mc["d_model"], _mc["n_layers"], _mc["d_state"], _mc["d_conv"],
                              _mc["expand"], _mc["dropout"]).eval()
_enc.return_sequence = True
for _L in (30, 60):
    _x = torch.randn(6, _L, 8)
    with torch.no_grad():
        _y1 = _enc(_x)
        for _t in (0, 3, _L // 2, _L - 2):
            _x2 = _x.clone()
            _x2[:, _t + 1:] = torch.randn_like(_x2[:, _t + 1:])
            _y2 = _enc(_x2)
            assert torch.allclose(_y1[:, :_t + 1], _y2[:, :_t + 1], atol=1e-6), \
                f"Mamba CAUSALITY VIOLATED (L={_L}, t={_t}): future days leak into past states"
UT["mamba_causality"] = "pass"

# 2. Mamba shape, determinism, stability
_enc.return_sequence = False
_enc.train()
for _L in (30, 60):
    _out = _enc(torch.randn(40, _L, 8))
    assert tuple(_out.shape) == (40, _mc["d_model"]), _out.shape
def _fp(seed):
    torch.manual_seed(seed)
    return d4_models.MambaEncoder(8, _mc["d_model"], _mc["n_layers"], _mc["d_state"], _mc["d_conv"],
                                  _mc["expand"], _mc["dropout"])
def _hash(m):
    h = hashlib.sha256()
    for p in m.parameters():
        h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
assert _hash(_fp(7)) == _hash(_fp(7)) and _hash(_fp(7)) != _hash(_fp(8))
_enc2 = _fp(1)
torch.manual_seed(1)
for _ in range(200):
    _o = _enc2(torch.randn(40, 30, 8))
    _o.pow(2).mean().backward()
    assert torch.isfinite(_o).all(), "NaN/Inf in Mamba output"
    assert all(torch.isfinite(p.grad).all() for p in _enc2.parameters() if p.grad is not None), "NaN/Inf grad"
    _enc2.zero_grad()
UT["mamba_shape_determinism_stability"] = "pass"

# 3. Incremental correlation == full recomputation on every date
_rets = d4_graph.universe_returns(cfg)
_W = CONFIG["graph"]["corr_window"]
_rc = d4_graph.RollingCorr(_W, _rets.shape[1], CONFIG["phase4b"]["full_recompute_every"])
_maxdiff, _n = 0.0, 0
for _i, _r in enumerate(_rets.to_numpy(dtype=np.float64)):
    _rc.push(_r)
    if _i >= _W - 1:
        _full = _rets.iloc[_i - _W + 1: _i + 1].corr().values
        _maxdiff = max(_maxdiff, float(np.abs(_rc.corr() - _full).max()))
        _n += 1
assert _maxdiff < 1e-9, f"RollingCorr deviates from full recomputation: max |diff| {_maxdiff:.3e}"
UT["rolling_corr_max_abs_diff"] = _maxdiff
UT["rolling_corr_dates_checked"] = _n

# 4. Daily policy reproduces graphs_baseline.parquet (≥ 99.5% of dates; mismatches = k-th boundary ties)
_edges = d4_graph.load_daily_edges()
_build = d4_graph.make_graph_builder(cfg)
_rows = d4_graph.return_rows(_rets)
_days = [pd.Timestamp(d) for d in _rets.index[_W:]]               # 2nd graph date onward
_rc4 = d4_graph.RollingCorr(_W, _rets.shape[1], CONFIG["phase4b"]["full_recompute_every"])
for _r in _rets.iloc[:_W].to_numpy(dtype=np.float64):
    _rc4.push(_r)
_graphs, _ = d4_graph.run_policy(_days, lambda d: _rows[d], _build, lambda d: 0.0,
                                 d4_graph.rule_daily, float("inf"), _rc4)
def _edge_set(g):
    return set(zip(g[1].tolist(), g[0][0].tolist(), g[0][1].tolist()))
_first = _rets.index[_W - 1]
_checked = [(pd.Timestamp(_first), _build(d4_graph.warm_rolling_corr(_rets, _first, _W).corr()))] + \
           [(d, _graphs[d]) for d in _days]
_bad = []
for _d, _g in _checked:
    _ref = _edges[_d]
    if _edge_set((( _g[0]), _g[1])) != _edge_set(((_ref[0]), _ref[1])):
        _bad.append(_d)
assert len(_bad) / len(_checked) <= 0.005, f"{len(_bad)}/{len(_checked)} dates differ from graphs_baseline.parquet"
_kpos, _kneg = CONFIG["graph"]["k_pos"], CONFIG["graph"]["k_neg"]
for _d in _bad:                                                   # every mismatch must be a boundary tie
    _i = _rets.index.get_loc(_d)
    _C = _rets.iloc[_i - _W + 1: _i + 1].corr().values.copy()
    _ours, _theirs = _edge_set(_graphs.get(_d, _build(_C))), _edge_set(_edges[_d])
    for (_t, _s, _dd) in (_ours ^ _theirs):
        if _t == 2:
            raise AssertionError(f"sector edge mismatch on {_d.date()}")
        _row = np.delete(_C[_s], _s)
        _v = np.sort(_row)[::-1] if _t == 0 else np.sort(_row)
        _k = _kpos if _t == 0 else _kneg
        assert abs(_v[_k - 1] - _v[_k]) < 1e-8, f"non-tie edge mismatch on {_d.date()} (type {_t}, stock {_s})"
UT["daily_policy_vs_baseline"] = {"dates_checked": len(_checked), "mismatching_dates": len(_bad),
                                  "match_fraction": 1 - len(_bad) / len(_checked)}

# 5. Data alignment + test gate
_tb = d4_data.tables()
_prices = pd.read_parquet(d4_data.resolve_cache() / "prices.parquet")
_close = _prices.pivot(index="date", columns="ticker", values="close").sort_index()[UNIVERSE]
_nxt = _close.shift(-1) / _close - 1
_ex = _nxt.sub(_nxt.mean(axis=1), axis=0)
for _fold in [f.name for f in _tb["folds"]]:
    for _lb in (30, 60):
        _sd = d4_graph.get_split_g(_fold, "train", _lb)
        _bld = d4_data.get_builder(_fold, _lb)
        assert _bld.scaler_fit_dates[1] <= _tb["folds"][[f.name for f in _tb["folds"]].index(_fold)].train_dates[-1], \
            "scaler fit saw dates beyond the training window"
        for _j in np.linspace(0, len(_sd.dates) - 1, 5).astype(int):
            _d = pd.Timestamp(_sd.dates[_j])
            _loc = _tb["feat_wide"].index.get_loc(_d)
            _last_feat_date = _tb["feat_wide"].index[_loc]          # window ends at d
            _target_date = _close.index[_close.index.get_loc(_d) + 1]
            assert _last_feat_date < _target_date, "feature window not strictly before the target date"
            assert np.allclose(_sd.y[_j], _ex.loc[_d].values, atol=1e-12), "target is not the next-day excess return"
            assert _sd.X.shape[1:] == (40, _lb, 8)
try:
    d4_data.get_split("fold_1_2019_calm", "test")
    raise AssertionError("test split was readable while locked")
except PermissionError:
    pass
UT["data_alignment_and_test_gate"] = "pass"

(WORK / "phase4_unit_tests.json").write_text(json.dumps(UT, indent=2))
print(json.dumps(UT, indent=2))
print(f"unit tests passed in {time.time() - _t0:.0f}s")


In [ ]:
# ─── Cell 11: leakage gate (validation data only; test is still locked) ───
import torch, d4_train, d4_eval

_t0 = time.time()
_max_se = CONFIG["leakage_gate"]["max_abs_ic_in_standard_errors"]

def _ic_stats(pred, y):
    ics = np.array([d4_eval.per_day_rank_ic(pred[i], y[i]) for i in range(len(pred))])
    ics = ics[np.isfinite(ics)]
    se = float(ics.std(ddof=1) / np.sqrt(len(ics)))
    return float(ics.mean()), se, len(ics), ics

GATE = {"max_abs_ic_in_standard_errors": _max_se}
_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_fold_names = [f["name"] for f in STATIC["folds"]]

# 1. Random-weights model (Mamba, lookback 30) through the full pipeline, all folds' validation data
_edges = d4_graph.load_daily_edges()
_all, _per_fold = [], {}
for _fn in _fold_names:
    d4_train.set_seed(0)
    _net = d4_train.build_model("B").to(_dev).eval()
    _sd = d4_graph.get_split_g(_fn, "val", CONFIG["arms"]["B"]["lookback"])
    _X = torch.from_numpy(_sd.X).float().to(_dev)
    _ei = [_edges[pd.Timestamp(d)][0].to(_dev) for d in _sd.dates]
    _et = [_edges[pd.Timestamp(d)][1].to(_dev) for d in _sd.dates]
    _pred = d4_train.predict(_net, _X, _ei, _et, 32)
    _m, _se, _n, _ics = _ic_stats(_pred, _sd.y)
    _per_fold[_fn] = {"mean_rank_ic": _m, "se": _se, "n_days": _n}
    _all.append(_ics)
_ics = np.concatenate(_all)
_m, _se = float(_ics.mean()), float(_ics.std(ddof=1) / np.sqrt(len(_ics)))
GATE["random_weights"] = {"mean_rank_ic": _m, "se": _se, "n_days": int(len(_ics)),
                          "abs_ic_in_se": abs(_m) / _se, "per_fold": _per_fold,
                          "passed": bool(abs(_m) < _max_se * _se)}

# 2. Shuffled-target run: arm B, fold 1, seed 0, training labels scrambled across dates
_res = d4_train.train_one("B", _fold_names[0], 0, CONFIG["training"]["batch_size_ladder"][0],
                          out_dir=WORK / "leakage_gate", shuffle_targets=True, write=False)
_sd = d4_graph.get_split_g(_fold_names[0], "val", CONFIG["arms"]["B"]["lookback"])
_m, _se, _n, _ = _ic_stats(_res["val_pred"], _sd.y)
GATE["shuffled_targets"] = {"mean_rank_ic": _m, "se": _se, "n_days": _n, "abs_ic_in_se": abs(_m) / _se,
                            "epochs_run": _res["epochs_run"], "passed": bool(abs(_m) < _max_se * _se)}

# 3. Mamba causality (unit test 1)
GATE["mamba_causality_unit_test"] = UT.get("mamba_causality") == "pass"

GATE["passed"] = bool(GATE["random_weights"]["passed"] and GATE["shuffled_targets"]["passed"]
                      and GATE["mamba_causality_unit_test"])
GATE["seconds"] = time.time() - _t0
(WORK / "phase4_leakage_gate.json").write_text(json.dumps(GATE, indent=2))
print(json.dumps(GATE, indent=2))
if not GATE["passed"]:
    raise RuntimeError("LEAKAGE GATE FAILED — see phase4_leakage_gate.json; ending the run before long training.")


In [ ]:
# ─── Cell 12: memory + timing probe → budget plan ───
import torch, d4_train

_B = CONFIG["budget"]
_ladder = CONFIG["training"]["batch_size_ladder"]
_fold_names = [f["name"] for f in STATIC["folds"]]
_big = _fold_names[-1]                                   # fold 4: the largest
PLAN = {"n_gpus": n_gpus, "probe_fold": _big}

# 1. Memory: heaviest arm (C) on the largest fold; same batch size for ALL arms
_mem_log, BATCH_SIZE = [], None
for _bs in _ladder:
    try:
        _peak = d4_train.probe_memory("C", _big, _bs)
        _mem_log.append({"batch_size": _bs, "peak_gb": _peak, "status": "ok"})
        if _peak > 12.0:
            _mem_log[-1]["status"] = "peak > 12 GB"
            continue
        BATCH_SIZE = _bs
        break
    except torch.cuda.OutOfMemoryError:
        _mem_log.append({"batch_size": _bs, "peak_gb": None, "status": "OOM"})
        torch.cuda.empty_cache()
if BATCH_SIZE is None:
    raise RuntimeError(f"No batch size in {_ladder} fits arm C on {_big}: {_mem_log}")
PLAN["memory_probe"], PLAN["batch_size"] = _mem_log, BATCH_SIZE
print("memory probe:", _mem_log, "→ batch size", BATCH_SIZE)

# 2. Timing: arms A, B, C on fold 4 / seed 0, 2 epochs, time epoch 2 (probe runs are discarded)
SEC_PER_EPOCH = {}
for _arm in ("A", "B", "C"):
    _r = d4_train.train_one(_arm, _big, 0, BATCH_SIZE, max_epochs=2, out_dir=WORK / "probe", write=False)
    SEC_PER_EPOCH[_arm] = _r["last_epoch_seconds"]
    print(f"arm {_arm}: {SEC_PER_EPOCH[_arm]:.1f}s per epoch (epoch 2)")
PLAN["sec_per_epoch"] = SEC_PER_EPOCH

# 3. Projection
def _n_train(arm, fold):
    return len(d4_graph.get_split_g(fold, "train", CONFIG["arms"][arm]["lookback"]).dates)

PER_RUN = {}
for _arm in ("A", "B", "C"):
    for _fn in _fold_names:
        PER_RUN[(_arm, _fn)] = (SEC_PER_EPOCH[_arm] * (_n_train(_arm, _fn) / _n_train(_arm, _big))
                                * _B["expected_epochs_for_projection"] * _B["safety_factor"])
for _fn in _fold_names:
    PER_RUN[("D", _fn)] = PER_RUN[("C", _fn)] * 1.05

def lpt_assign(jobs, n_gpus):
    """Longest-job-first onto the GPU with the least assigned time. jobs: [(dur, job)]."""
    loads, assign = [0.0] * n_gpus, [[] for _ in range(n_gpus)]
    for dur, job in sorted(jobs, key=lambda x: -x[0]):
        k = int(np.argmin(loads))
        loads[k] += dur
        assign[k].append(job)
    return assign, loads

def _jobs(arms):
    return [(PER_RUN[(a, f)], {"arm": a, "fold": f, "seed": s})
            for a in arms for f in _fold_names for s in CONFIG["seeds"]]

_elapsed_h = (time.time() - RUN_T0) / 3600
_tail_h = _B["phase4b_estimate_minutes"] / 60 + _B["scoring_and_bench_estimate_minutes"] / 60
PROJ = {}
for _name, _arms in (("with_D", ["A", "B", "C", "D"]), ("without_D", ["A", "B", "C"])):
    _, _loads = lpt_assign(_jobs(_arms), n_gpus)
    PROJ[_name] = {"makespan_h": max(_loads) / 3600, "n_jobs": len(_jobs(_arms)),
                   "projected_total_h": _elapsed_h + max(_loads) / 3600 + _tail_h}
PLAN["projection"] = PROJ
PLAN["per_run_seconds"] = {f"{a}|{f}": v for (a, f), v in PER_RUN.items()}

# 4. Decision (pre-registered, automatic; seeds / folds / epochs are never reduced)
_limit = _B["plan_limit_hours"]
RUN_D = False
if PROJ["with_D"]["projected_total_h"] <= _limit:
    RUN_D = True
elif PROJ["without_D"]["projected_total_h"] <= _limit:
    PLAN["D_skipped_reason"] = "budget"
else:
    (WORK / "budget_plan.json").write_text(json.dumps(PLAN, indent=2))
    print(json.dumps(PROJ, indent=2))
    raise RuntimeError(f"Arms A+B+C do not fit the {_limit} h plan limit: {PROJ['without_D']}")
PLAN["run_D"] = RUN_D
PLAN["arms_run"] = ["A", "B", "C"] + (["D"] if RUN_D else [])
(WORK / "budget_plan.json").write_text(json.dumps(PLAN, indent=2))
print(json.dumps({k: PLAN[k] for k in ("batch_size", "projection", "run_D", "arms_run")}, indent=2))


In [ ]:
# ─── Cell 13: launch all training jobs across GPUs, wait, completeness check ───
import subprocess, sys, glob

_fold_names = [f["name"] for f in STATIC["folds"]]
ARMS_RUN = PLAN["arms_run"]
_jobs_all = _jobs(ARMS_RUN)                                  # 60 (without D) or 80 (with D) jobs
_assign, _loads = lpt_assign(_jobs_all, n_gpus)
(WORK / "logs").mkdir(exist_ok=True)
for _k in range(n_gpus):
    (WORK / f"jobs_gpu{_k}.json").write_text(json.dumps(_assign[_k], indent=2))
print(f"{len(_jobs_all)} jobs on {n_gpus} GPU(s); projected makespan "
      f"{max(_loads) / 3600:.2f} h; batch size {BATCH_SIZE}")

_procs = []
for _k in range(n_gpus):
    _env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(_k), CUBLAS_WORKSPACE_CONFIG=":4096:8",
                DMRGT4_WORK=str(WORK), DMRGT4_CACHE=str(CACHE))
    _log = open(WORK / "logs" / f"gpu{_k}.log", "w")
    _procs.append(subprocess.Popen(
        [sys.executable, "/kaggle/working/dmrgt4/d4_worker.py", "--jobs", str(WORK / f"jobs_gpu{_k}.json"),
         "--batch-size", str(BATCH_SIZE)], env=_env, stdout=_log, stderr=subprocess.STDOUT))

def _n_done():
    return len(glob.glob(str(WORK / "done_*.json")))

_t_launch = time.time()
while any(p.poll() is None for p in _procs):
    time.sleep(60)
    _d, _el = _n_done(), time.time() - _t_launch
    _eta = (_el / _d * (len(_jobs_all) - _d)) if _d else float("nan")
    print(f"[{_el / 60:6.1f} min] {_d}/{len(_jobs_all)} jobs done; projected remaining {_eta / 60:.0f} min", flush=True)
TRAIN_SECONDS = time.time() - _t_launch

# completeness: A, B, C need all 20 markers; D is eligible only if complete
def _arm_done(arm):
    return [(f, s) for f in _fold_names for s in CONFIG["seeds"]
            if d4_train.ckpt_paths(WORK, arm, f, s)["done"].exists()]

_n_expected = len(_fold_names) * len(CONFIG["seeds"])
COMPLETE = {a: len(_arm_done(a)) == _n_expected for a in ARMS_RUN}
if not all(COMPLETE[a] for a in ("A", "B", "C")):
    for _p in sorted(glob.glob(str(WORK / "error_*.json"))):
        print(_p, "\n", json.loads(Path(_p).read_text())["traceback"])
    raise RuntimeError(f"Required arms incomplete: { {a: len(_arm_done(a)) for a in ARMS_RUN} } of {_n_expected}")
if "D" in ARMS_RUN and not COMPLETE["D"]:
    print(f"Arm D incomplete ({len(_arm_done('D'))}/{_n_expected}) — excluded from selection, reported as such.")
print("completeness:", COMPLETE, f"| training wall time {TRAIN_SECONDS / 3600:.2f} h")


In [ ]:
# ─── Cell 14: Phase 4A selection — validation only ───
import hashlib, d4_eval

TARGETS = pd.read_parquet(CACHE / "targets.parquet")
_fold_names = [f["name"] for f in STATIC["folds"]]
_seeds = CONFIG["seeds"]

def load_arm_preds(arm):
    frames = [pd.read_parquet(d4_train.ckpt_paths(WORK, arm, f, s)["valpred"])
              for f in _fold_names for s in _seeds]
    return pd.concat(frames, ignore_index=True)

_rows, _val_days = [], None
for _arm in PLAN["arms_run"]:
    if not COMPLETE[_arm]:
        continue
    _df = load_arm_preds(_arm)
    assert set(_df["split"].unique()) == {"val"}, "valpred files must hold validation rows only"
    # row count: SUM of per-fold day counts (not a product) × seeds × stocks
    _n_days = {f: _df[_df["fold"] == f]["date"].nunique() for f in _fold_names}
    assert len(_df) == sum(_n_days.values()) * len(_seeds) * len(UNIVERSE), f"{_arm}: valpred row count mismatch"
    _res = d4_eval.evaluate_predictions(_df, TARGETS, cfg, split="val")
    assert len(_res) == len(_fold_names) * len(_seeds)
    _rows.append({"arm": _arm, "encoder": CONFIG["arms"][_arm]["encoder"], "lookback": CONFIG["arms"][_arm]["lookback"],
                  "adaptive_huber": CONFIG["arms"][_arm]["adaptive_huber"], "n_runs": len(_res),
                  "val_sharpe": float(np.mean([r.fold_metrics.sharpe_after_costs for r in _res])),
                  "val_mean_rank_ic": float(np.mean([r.fold_metrics.mean_rank_ic for r in _res])),
                  "val_dir_acc": float(np.mean([r.fold_metrics.directional_acc for r in _res])),
                  "val_turnover": float(np.mean([r.fold_metrics.turnover for r in _res]))})
VAL4A = pd.DataFrame(_rows)
VAL4A.to_csv(WORK / "phase4_validation_summary.csv", index=False)
print(VAL4A.to_string(index=False))

_sel = CONFIG["selection_4A"]
_cand = VAL4A[VAL4A["arm"].isin(_sel["candidates"])].sort_values("val_sharpe", ascending=False).reset_index(drop=True)
assert len(_cand) >= 2, "need at least B and C complete to select a winner"
WINNER = _cand.loc[0, "arm"]
_tie = False
if abs(_cand.loc[0, "val_sharpe"] - _cand.loc[1, "val_sharpe"]) < _sel["tie_threshold"]:
    _tie = True
    if _cand.loc[1, "val_mean_rank_ic"] > _cand.loc[0, "val_mean_rank_ic"]:
        WINNER = _cand.loc[1, "arm"]
print(f"Phase 4A winner: {WINNER} (tiebreak applied: {_tie})")

_ckpt_hashes = {f"{f}|{s}": hashlib.sha256(d4_train.ckpt_paths(WORK, WINNER, f, s)["ckpt"].read_bytes()).hexdigest()
                for f in _fold_names for s in _seeds}
(WORK / "phase4_winner.json").write_text(json.dumps({
    "winner_arm": WINNER, "arm_config": CONFIG["arms"][WINNER], "tiebreak_applied": _tie,
    "selection_rule": _sel, "validation_summary": VAL4A.to_dict(orient="records"),
    "checkpoint_sha256": _ckpt_hashes, "config_sha256": CONFIG_SHA256,
    "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
print("wrote phase4_winner.json (test still locked until phase4b_policy.json exists)")


In [ ]:
# ─── Cell 15: Phase 4B — trigger calibration (train) + policy selection (validation) ───
# Inference only: the winner's 20 checkpoints; nothing is trained. Test is still locked.
import d4_graph, d4_eval

_t0 = time.time()
_P = CONFIG["phase4b"]
_W, _MAXSTALE = _P["corr_window"], _P["max_staleness_days"]
_folds = {f.name: f for f in d4_data.tables()["folds"]}
_fold_names = [f["name"] for f in STATIC["folds"]]
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_encoder = CONFIG["arms"][WINNER]["encoder"]
RETS = d4_graph.universe_returns(cfg)
CORR = d4_graph.corr_by_date(RETS, _W)
_sigs = [s for s in _P["signals"] if _encoder == "mamba" or s == "corr_drift"]   # Δ needs a Mamba encoder

TRIGGERS = [{"name": f"{s}@{int(round(r * 100))}", "kind": "trigger", "signal": s, "target_rate": r}
            for s in _sigs for r in _P["target_rebuild_rates"]]
DAILY = {"name": "daily", "kind": "daily"}
FROZEN = {"name": "frozen_at_start", "kind": "frozen"}
_all_pol = TRIGGERS + [DAILY, FROZEN]

INF = {}                                    # (fold, seed) → ModelInference, reused by Cell 16 / 17
THRESH = {}                                 # "fold|seed" → {policy: thresholds}
_frames = {p["name"]: [] for p in _all_pol}
_rates = {p["name"]: [] for p in _all_pol}  # (n_rebuilds, n_days) per run
_corr_cache = {}                            # per-fold corr-threshold memo (state-independent of the model)

for _fn in _fold_names:
    _fold = _folds[_fn]
    for _seed in CONFIG["seeds"]:
        _mi = d4_graph.ModelInference(d4_train.ckpt_paths(WORK, WINNER, _fn, _seed)["ckpt"], _DEV, SNAPSHOT)
        INF[(_fn, _seed)] = _mi
        _dtrain = _mi.delta_by_day(["train"]) if _encoder == "mamba" else None
        _dval = _mi.delta_by_day(["val"]) if _encoder == "mamba" else None
        # calibration sim over the fold's TRAINING days (leakage-free)
        _days = [pd.Timestamp(d) for d in _mi.split_data("train").dates]
        _pos = RETS.index.get_indexer(_days)
        assert (np.diff(_pos) == 1).all(), "training sample dates are not contiguous in the return calendar"
        _steps = [CORR[RETS.index[_pos[0] - 1]]] + [CORR[d] for d in _days]
        THRESH[f"{_fn}|{_seed}"] = {}
        for _p in TRIGGERS:
            _th = d4_graph.calibrate_thresholds(_p["signal"], _p["target_rate"], _days, _steps, _dtrain,
                                                _MAXSTALE, corr_cache=_corr_cache.setdefault(_fn, {}))
            THRESH[f"{_fn}|{_seed}"][_p["name"]] = _th
        # validation: each policy runs continuously from the last training day through validation
        for _p in _all_pol:
            _g, _rb, _st = d4_graph.run_named_policy(
                _p, THRESH[f"{_fn}|{_seed}"].get(_p["name"]), _fold, RETS, _fold.val_dates[-1],
                _dval, cfg, _W, _MAXSTALE)
            _pred = _mi.predict_with_graphs("val", _g)
            _frames[_p["name"]].append(_mi.frame(_p["name"], "val", _pred))
            _rates[_p["name"]].append((_st["n_rebuilds"], _st["n_days"]))
    print(f"fold {_fn} done ({time.time() - _t0:.0f}s)", flush=True)

_targets = TARGETS
_rows = []
for _p in _all_pol:
    _df = pd.concat(_frames[_p["name"]], ignore_index=True)
    _bt = d4_eval.backtest(_df, cfg, split="val", targets=_targets)
    _nr, _nd = np.sum([r[0] for r in _rates[_p["name"]]]), np.sum([r[1] for r in _rates[_p["name"]]])
    _rows.append({"policy": _p["name"], "kind": _p["kind"], "signal": _p.get("signal"),
                  "target_rate": _p.get("target_rate"), "val_rebuild_rate": float(_nr / _nd),
                  "val_sharpe": _bt["sharpe"], "val_mean_rank_ic": _bt["mean_rank_ic"],
                  "val_turnover": _bt["turnover"], "n_runs": _bt["n_runs"]})
VAL4B = pd.DataFrame(_rows)
VAL4B.to_csv(WORK / "phase4b_validation_summary.csv", index=False)
print(VAL4B.to_string(index=False))

# integration check: the incremental `daily` policy must reproduce the arm's own validation Sharpe
_daily_sh = float(VAL4B.loc[VAL4B["policy"] == "daily", "val_sharpe"].iloc[0])
_arm_sh = float(VAL4A.loc[VAL4A["arm"] == WINNER, "val_sharpe"].iloc[0])
assert abs(_daily_sh - _arm_sh) < 0.05, (
    f"daily-policy val Sharpe {_daily_sh:.4f} != winner's Phase 4A val Sharpe {_arm_sh:.4f}")

# selection rule (pre-registered): lowest realised val rebuild rate among the trigger policies whose
# mean val Sharpe >= daily's − margin; ties → higher val rank IC; none qualifies → highest val Sharpe.
_margin = _P["val_noninferiority_margin_sharpe"]
_trig = VAL4B[VAL4B["kind"] == "trigger"]
_ok = _trig[_trig["val_sharpe"] >= _daily_sh - _margin]
if len(_ok):
    _best = _ok.sort_values(["val_rebuild_rate", "val_mean_rank_ic"], ascending=[True, False]).iloc[0]
    _noninf = True
else:
    _best = _trig.sort_values("val_sharpe", ascending=False).iloc[0]
    _noninf = False
SELECTED = next(p for p in TRIGGERS if p["name"] == _best["policy"])
print(f"Phase 4B selected policy: {SELECTED['name']} (val rebuild rate {_best['val_rebuild_rate']:.3f}, "
      f"val Sharpe {_best['val_sharpe']:.3f} vs daily {_daily_sh:.3f}; non-inferiority met: {_noninf})")

(WORK / "phase4b_policy.json").write_text(json.dumps({
    "policy": SELECTED, "winner_arm": WINNER, "max_staleness_days": _MAXSTALE, "corr_window": _W,
    "thresholds": {k: v[SELECTED["name"]] for k, v in THRESH.items()},
    "all_thresholds": THRESH,
    "realised_val_rebuild_rate": float(_best["val_rebuild_rate"]),
    "val_sharpe_selected": float(_best["val_sharpe"]), "val_sharpe_daily": _daily_sh,
    "val_margin": _margin, "validation_noninferiority_met": _noninf,
    "daily_vs_phase4A_val_sharpe_diff": _daily_sh - _arm_sh,
    "validation_table": VAL4B.to_dict(orient="records"),
    "config_sha256": CONFIG_SHA256,
    "selected_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
PHASE4B_SECONDS = time.time() - _t0
print(f"wrote phase4b_policy.json — test split is now unlocked ({PHASE4B_SECONDS / 60:.1f} min)")


In [ ]:
# ─── Cell 16: ONE-SHOT test scoring (runs only because both lock files now exist) ───
import hashlib, d4_stats
from d4_eval import random_model, persistence_model

if (WORK / "phase4_TEST_SCORED.json").exists():
    raise RuntimeError("phase4_TEST_SCORED.json already exists — no double scoring within a run.")
d4_data.assert_test_unlocked()
_t0 = time.time()
_folds = {f.name: f for f in d4_data.tables()["folds"]}
_fold_names = [f["name"] for f in STATIC["folds"]]
_seeds = CONFIG["seeds"]
_edges = d4_graph.load_daily_edges()
_thr = json.loads((WORK / "phase4b_policy.json").read_text())["thresholds"]

_pred_hashes, TEST = {}, {}

def _save_testpred(name, fold, seed, df):
    p = WORK / f"testpred_{name}_{fold}_{seed}.parquet"
    df.to_parquet(p, index=False)
    _pred_hashes[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()

# ── 4A: every completed arm, all 20 checkpoints, the cached daily graph ──
for _arm in PLAN["arms_run"]:
    if not COMPLETE[_arm]:
        continue
    _fr = []
    for _fn in _fold_names:
        for _s in _seeds:
            _mi = INF.get((_fn, _s)) if _arm == WINNER else None
            if _mi is None:
                _mi = d4_graph.ModelInference(d4_train.ckpt_paths(WORK, _arm, _fn, _s)["ckpt"], _DEV, SNAPSHOT)
            _df = _mi.frame(_arm, "test", _mi.predict_with_graphs("test", _edges))
            _save_testpred(_arm, _fn, _s, _df)
            _fr.append(_df)
    TEST[_arm] = pd.concat(_fr, ignore_index=True)

# ── 4B: the winner's checkpoints under selected / daily / frozen_at_start (continuous from the last training day) ──
_sel = json.loads((WORK / "phase4b_policy.json").read_text())["policy"]
_pols = {f"{WINNER}_selected": _sel, f"{WINNER}_daily": DAILY, f"{WINNER}_frozen_at_start": FROZEN}
POLICY_STATS = {n: [] for n in _pols}
for _fn in _fold_names:
    _fold = _folds[_fn]
    _test_set = set(pd.Timestamp(d) for d in d4_graph.get_split_g(_fn, "test", CONFIG["arms"][WINNER]["lookback"]).dates)
    for _s in _seeds:
        _mi = INF[(_fn, _s)]
        _dby = _mi.delta_by_day(["val", "test"]) if _encoder == "mamba" else None
        for _n, _p in _pols.items():
            _g, _rb, _st = d4_graph.run_named_policy(_p, _thr[f"{_fn}|{_s}"] if _p["kind"] == "trigger" else None,
                                                     _fold, RETS, _fold.test_dates[-1], _dby, cfg, _W, _MAXSTALE)
            _df = _mi.frame(_n, "test", _mi.predict_with_graphs("test", _g))
            _save_testpred(_n, _fn, _s, _df)
            _n_test_days = len([d for d in d4_graph.policy_days(RETS, _fold, _fold.test_dates[-1]) if d in _test_set])
            POLICY_STATS[_n].append({"fold": _fn, "seed": _s, "test_rebuilds": sum(1 for d in _rb if d in _test_set),
                                     "test_days": _n_test_days, "graph_build_seconds": _st["graph_build_seconds"],
                                     "policy_seconds": _st["policy_seconds"]})
            TEST.setdefault(_n, []).append(_df)
for _n in _pols:
    TEST[_n] = pd.concat(TEST[_n], ignore_index=True)

# ── metrics: every model through the same harness, mean over its 20 (fold, seed) runs ──
_rows = []
for _name, _df in TEST.items():
    _res = d4_eval.evaluate_predictions(_df, TARGETS, cfg, split="test")
    _m = pd.DataFrame([vars(r.fold_metrics) for r in _res])
    _rows.append({"model": _name, "n_runs": len(_m), **{k: float(_m[k].mean()) for k in
                  ("mean_rank_ic", "directional_acc", "sharpe_after_costs", "max_drawdown", "turnover", "cum_return")}})
# floors: random + persistence through the same harness (same 4 folds × 5 seeds)
_feats, _tgts = d4_data.tables()["features"], d4_data.tables()["targets"]
for _name, _fn_ in (("random", lambda s: (lambda X: random_model(X, seed=s))),
                    ("persistence", lambda s: (lambda X: persistence_model(X)))):
    _ms = [d4_eval.evaluate_fold(_name, _fn_(s), _folds[f], _feats, _tgts, cfg, seed=s).fold_metrics
           for f in _fold_names for s in _seeds]
    _m = pd.DataFrame([vars(x) for x in _ms])
    _rows.append({"model": _name, "n_runs": len(_m), **{k: float(_m[k].mean()) for k in
                  ("mean_rank_ic", "directional_acc", "sharpe_after_costs", "max_drawdown", "turnover", "cum_return")}})
TEST_METRICS = pd.DataFrame(_rows)
TEST_METRICS.to_csv(WORK / "phase4_test_metrics.csv", index=False)
print(TEST_METRICS.to_string(index=False))

# ── significance ──
_alpha = CONFIG["significance"]["alpha"]
SIG = {"phase4a": {}, "phase4b": {}}
SIG["phase4a"]["primary"] = {"comparison": f"{WINNER} vs A", **d4_stats.compare_arms(TEST[WINNER], TEST["A"], cfg)}
_others = [a for a in ("B", "C", "D") if a in TEST and a != WINNER]
_sec = {a: d4_stats.compare_arms(TEST[a], TEST["A"], cfg) for a in _others}
SIG["phase4a"]["secondary"] = {f"{a} vs A": v for a, v in _sec.items()}
if _sec:
    SIG["phase4a"]["secondary_holm_wilcoxon"] = d4_stats.holm_correction(
        {f"{a} vs A": v["wilcoxon"]["p_value"] for a, v in _sec.items()}, _alpha)
    SIG["phase4a"]["secondary_holm_bootstrap_two_sided"] = d4_stats.holm_correction(
        {f"{a} vs A": v["bootstrap_sharpe"]["p_two_sided"] for a, v in _sec.items()}, _alpha)

_selN, _dailyN, _frozN = f"{WINNER}_selected", f"{WINNER}_daily", f"{WINNER}_frozen_at_start"
try:
    _vs_daily = d4_stats.compare_arms(TEST[_selN], TEST[_dailyN], cfg)
except ValueError as _e:                    # identical per-run ICs → Wilcoxon undefined
    _a, _b = d4_stats._test_rows(TEST[_selN]), d4_stats._test_rows(TEST[_dailyN])
    _vs_daily = {"wilcoxon": {"note": f"Wilcoxon undefined: {_e}", "p_value": 1.0},
                 "bootstrap_sharpe": d4_stats.bootstrap_sharpe_test(
                     d4_stats.daily_returns(_a, cfg), d4_stats.daily_returns(_b, cfg), cfg)}
_margin = CONFIG["phase4b"]["test_noninferiority_margin_sharpe"]
_lo = _vs_daily["bootstrap_sharpe"]["ci_95"][0]
_vs_daily["noninferiority"] = {"margin": _margin, "ci95_lower_bound": _lo, "non_inferior": bool(_lo > -_margin)}
SIG["phase4b"]["selected_vs_daily"] = _vs_daily
SIG["phase4b"]["selected_vs_frozen_at_start"] = d4_stats.compare_arms(TEST[_selN], TEST[_frozN], cfg)
_ps = pd.DataFrame([dict(policy=n, **r) for n, rs in POLICY_STATS.items() for r in rs])
SIG["phase4b"]["test_rebuild_rate"] = {n: float(g["test_rebuilds"].sum() / g["test_days"].sum())
                                       for n, g in _ps.groupby("policy")}
SIG["phase4b"]["graph_build_seconds_mean_per_run"] = {n: float(g["graph_build_seconds"].mean())
                                                      for n, g in _ps.groupby("policy")}
SIG["winner_arm"], SIG["selected_policy"] = WINNER, _sel["name"]
(WORK / "phase4_significance_results.json").write_text(json.dumps(SIG, indent=2, default=float))

(WORK / "phase4_TEST_SCORED.json").write_text(json.dumps({
    "winner_arm": WINNER, "policy": _sel, "config_sha256": CONFIG_SHA256,
    "prediction_file_sha256": _pred_hashes,
    "scored_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
TEST_SECONDS = time.time() - _t0
print(f"test scoring done in {TEST_SECONDS / 60:.1f} min")
print(json.dumps({"primary": {k: SIG["phase4a"]["primary"][k] for k in ("comparison",)},
                  "wilcoxon_p": SIG["phase4a"]["primary"]["wilcoxon"]["p_value"],
                  "bootstrap_ci95": SIG["phase4a"]["primary"]["bootstrap_sharpe"]["ci_95"],
                  "4B_noninferior_vs_daily": _vs_daily["noninferiority"],
                  "test_rebuild_rate": SIG["phase4b"]["test_rebuild_rate"]}, indent=2, default=float))


In [ ]:
# ─── Cell 17: efficiency + technical-effect measurements ───
import torch, d4_graph, d4_models

_t0 = time.time()
_fold_names = [f["name"] for f in STATIC["folds"]]
_seeds = CONFIG["seeds"]

# 14.1 Model efficiency — same GPU, same batch size, every completed arm (A vs winner is the headline pair)
def _done(arm):
    return [json.loads(d4_train.ckpt_paths(WORK, arm, f, s)["done"].read_text()) for f in _fold_names for s in _seeds]

def _latency_ms(arm, n_passes=200):
    """Median wall time of one single-day inference pass (encode + propagate), synchronised."""
    a = CONFIG["arms"][arm]
    d4_train.set_seed(0)
    net = d4_train.build_model(arm).to(_DEV).eval()
    sd = d4_graph.get_split_g(_fold_names[-1], "val", a["lookback"])
    x = torch.from_numpy(sd.X[:1]).float().to(_DEV)
    ei, et = (t.to(_DEV) for t in _edges[pd.Timestamp(sd.dates[0])])
    times = []
    with torch.no_grad():
        for i in range(n_passes + 10):
            if _DEV.type == "cuda":
                torch.cuda.synchronize()
            t = time.perf_counter()
            net(x, ei, et)
            if _DEV.type == "cuda":
                torch.cuda.synchronize()
            if i >= 10:                                  # discard warm-up passes
                times.append((time.perf_counter() - t) * 1000)
    return float(np.median(times))

_rows = []
for _arm in PLAN["arms_run"]:
    if not COMPLETE[_arm]:
        continue
    _d = _done(_arm)
    _rows.append({"arm": _arm, "encoder": CONFIG["arms"][_arm]["encoder"], "lookback": CONFIG["arms"][_arm]["lookback"],
                  "parameters": _d[0]["n_params"],
                  "mean_seconds_per_epoch": float(np.mean([x["mean_epoch_seconds"] for x in _d])),
                  "mean_epochs_to_early_stop": float(np.mean([x["epochs_run"] for x in _d])),
                  "mean_total_train_seconds_per_run": float(np.mean([x["total_seconds"] for x in _d])),
                  "inference_latency_ms_per_day_median": _latency_ms(_arm),
                  "peak_gpu_gb_max": float(np.max([x["peak_gpu_gb"] for x in _d])),
                  "batch_size": BATCH_SIZE})
EFF = pd.DataFrame(_rows)
EFF.to_csv(WORK / "phase4_efficiency.csv", index=False)
print(EFF.to_string(index=False))

# 14.2 Graph-construction cost (CPU, float64) — real 40-stock data
_build = d4_graph.make_graph_builder(cfg)
_sec_src, _sec_dst = d4_graph._sector_edges(cfg)
_fold4 = [f for f in d4_data.tables()["folds"] if f.name == _fold_names[-1]][0]
_idx = RETS.index
_pos_test = [_idx.get_loc(d) for d in _fold4.test_dates if d in _idx][:300]
_rc = d4_graph.warm_rolling_corr(RETS, _idx[_pos_test[0] - 1], _W)
_rows_ret = RETS.to_numpy(dtype=np.float64)
_t_full, _t_inc, _t_edges = [], [], []
for _i in _pos_test:
    t = time.perf_counter(); _C_full = RETS.iloc[_i - _W + 1: _i + 1].corr().values; _t_full.append(time.perf_counter() - t)
    t = time.perf_counter(); _rc.push(_rows_ret[_i]); _C_inc = _rc.corr(); _t_inc.append(time.perf_counter() - t)
    t = time.perf_counter(); _build(_C_inc); _t_edges.append(time.perf_counter() - t)
REAL_COST = {"n_updates_timed": len(_pos_test),
             "full_recompute_ms_median": float(np.median(_t_full) * 1000),
             "incremental_update_ms_median": float(np.median(_t_inc) * 1000),
             "edge_build_ms_median": float(np.median(_t_edges) * 1000),
             "policy_run_seconds_mean_per_run": SIG["phase4b"]["graph_build_seconds_mean_per_run"],
             "test_rebuild_rate": SIG["phase4b"]["test_rebuild_rate"]}
_n_td = len(_fold4.test_dates)
REAL_COST["test_period_total_seconds_full_recompute_daily"] = _n_td * (np.median(_t_full) + np.median(_t_edges))
REAL_COST["test_period_total_seconds_incremental_daily"] = _n_td * (np.median(_t_inc) + np.median(_t_edges))
REAL_COST["test_period_total_seconds_incremental_selected"] = _n_td * (
    np.median(_t_inc) + np.median(_t_edges) * SIG["phase4b"]["test_rebuild_rate"][f"{WINNER}_selected"])
(WORK / "phase4_real_graph_cost.json").write_text(json.dumps(REAL_COST, indent=2, default=float))
print(json.dumps(REAL_COST, indent=2, default=float))

# Synthetic scaling benchmark: random returns with a mid-sample regime switch, window 60, 750 days
_target_rate = float(SIG["phase4b"]["test_rebuild_rate"][f"{WINNER}_selected"])
_sec_none = ([], [])
def _bench(N, T=750, seed=0):
    rng = np.random.default_rng(seed)
    nf = max(3, N // 10)
    L1, L2 = rng.normal(size=(nf, N)), rng.normal(size=(nf, N))
    F = rng.normal(size=(T, nf))
    R = np.where((np.arange(T) < T // 2)[:, None], F @ L1, F @ L2) * 0.01 + rng.normal(size=(T, N)) * 0.01
    days = list(range(_W, T))
    bld = lambda C: d4_graph.edges_from_corr(C, cfg, [], [])
    t = time.perf_counter()
    for d in days:
        bld(np.corrcoef(R[d - _W + 1: d + 1].T))
    full = time.perf_counter() - t
    def run(rule, stale):
        rc = d4_graph.RollingCorr(_W, N)
        for r in R[:_W]:
            rc.push(r)
        t = time.perf_counter()
        _, rb = d4_graph.run_policy(days, lambda d: R[d], bld, lambda d: 0.0, rule, stale, rc)
        return time.perf_counter() - t, len(rb) / len(days)
    inc, _ = run(d4_graph.rule_daily, float("inf"))
    # corr-drift threshold bisected so the realised rate matches the selected policy's test rebuild rate
    pc_steps = [None] * (len(days) + 1)
    rc0 = d4_graph.RollingCorr(_W, N)
    for r in R[:_W]:
        rc0.push(r)
    pc_steps[0] = rc0.corr()
    for k, d in enumerate(days):
        rc0.push(R[d]); pc_steps[k + 1] = rc0.corr()
    def rate_of(th):
        return d4_graph.simulate_rate(days, pc_steps, None, d4_graph.make_rule("corr_drift", th, None), _MAXSTALE)
    th = d4_graph._bisect(rate_of, 0.0, 5.0, _target_rate)
    trig, real = run(d4_graph.make_rule("corr_drift", th, None), _MAXSTALE)
    return {"n_stocks": N, "days": len(days), "full_recompute_daily_s": full, "incremental_daily_s": inc,
            "incremental_trigger_s": trig, "trigger_target_rate": _target_rate, "trigger_realised_rate": real,
            "speedup_incremental_vs_full": full / inc, "speedup_trigger_vs_full": full / trig}

BENCH = pd.DataFrame([_bench(N) for N in CONFIG["phase4b"]["scale_benchmark_n_stocks"]])
BENCH.to_csv(WORK / "phase4b_compute_benchmark.csv", index=False)
print(BENCH.to_string(index=False))

# 14.3 Explainability hook: mean Δ per trading day (winner, averaged over seeds), 2019–2022. Read-only.
_drows = []
if _encoder == "mamba":
    for _fn in _fold_names:
        _per_seed = [pd.Series(INF[(_fn, s)].delta_by_day(["test"])) for s in _seeds]
        _dfm = pd.concat(_per_seed, axis=1)
        _drows.append(pd.DataFrame({"date": _dfm.index, "fold": _fn, "mean_delta": _dfm.mean(axis=1).values,
                                    "std_delta_across_seeds": _dfm.std(axis=1).values}))
DELTA = pd.concat(_drows, ignore_index=True) if _drows else pd.DataFrame(columns=["date", "fold", "mean_delta"])
DELTA.to_csv(WORK / "phase4_delta_by_date.csv", index=False)
if len(DELTA):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _fig, _ax = plt.subplots(figsize=(11, 3.2))
    _d = DELTA.sort_values("date")
    _ax.plot(pd.to_datetime(_d["date"]), _d["mean_delta"], lw=0.8)
    _ax.set_ylabel("mean Δ (memory-reset strength)"); _ax.set_title("Mamba Δ by trading day (winner, mean over seeds)")
    _fig.tight_layout(); _fig.savefig(WORK / "phase4_delta_by_date.png", dpi=130); plt.close(_fig)
EFF_SECONDS = time.time() - _t0
print(f"efficiency + technical-effect measurements done in {EFF_SECONDS / 60:.1f} min")


In [ ]:
# ─── Cell 18: output verification + auto-generated PHASE4_RESULTS_DRAFT.md (hard stop on failure) ───
import glob, hashlib

_fold_names = [f["name"] for f in STATIC["folds"]]
_seeds = CONFIG["seeds"]
_n_runs = len(_fold_names) * len(_seeds)
_required = ["config_snapshot.json", "run_manifest.json", "budget_plan.json", "phase4_leakage_gate.json",
             "phase4_validation_summary.csv", "phase4_winner.json",
             "phase4b_validation_summary.csv", "phase4b_policy.json",
             "phase4_test_metrics.csv", "phase4_significance_results.json", "phase4_TEST_SCORED.json",
             "phase4_efficiency.csv", "phase4b_compute_benchmark.csv", "phase4_delta_by_date.csv"]
_missing = [f for f in _required if not (WORK / f).exists()]
assert not _missing, f"missing required outputs: {_missing}"
_done_arms = [a for a in PLAN["arms_run"] if COMPLETE[a]]
for _a in _done_arms:
    assert len(glob.glob(str(WORK / f"ckpt_{_a}_*.pt"))) == _n_runs, f"arm {_a}: expected {_n_runs} checkpoints"
    assert len(glob.glob(str(WORK / f"log_{_a}_*.csv"))) == _n_runs, f"arm {_a}: expected {_n_runs} logs"
    assert len(glob.glob(str(WORK / f"valpred_{_a}_*.parquet"))) == _n_runs
    assert len(glob.glob(str(WORK / f"testpred_{_a}_fold_*.parquet"))) == _n_runs
assert len(glob.glob(str(WORK / "testpred_*.parquet"))) >= len(_done_arms) * _n_runs + 3 * _n_runs
assert len(glob.glob(str(WORK / "dmrgt4" / "*.py"))) >= 7, "dmrgt4/*.py (the code that produced the results) missing"
# the scored predictions on disk must still hash to what phase4_TEST_SCORED.json recorded
_scored = json.loads((WORK / "phase4_TEST_SCORED.json").read_text())
for _name, _h in _scored["prediction_file_sha256"].items():
    assert hashlib.sha256((WORK / _name).read_bytes()).hexdigest() == _h, f"{_name} changed after scoring"
print("all required outputs present and consistent")

# ── draft results, every number read from the files ──
def _md(df, floatfmt="{:.4f}"):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                      for v in r.values) + " |")
    return "\n".join(out)

_J = lambda n: json.loads((WORK / n).read_text())
_winner, _pol, _sig = _J("phase4_winner.json"), _J("phase4b_policy.json"), _J("phase4_significance_results.json")
_gate, _plan, _real = _J("phase4_leakage_gate.json"), _J("budget_plan.json"), _J("phase4_real_graph_cost.json")
_val4a, _val4b = pd.read_csv(WORK / "phase4_validation_summary.csv"), pd.read_csv(WORK / "phase4b_validation_summary.csv")
_tm, _eff, _bench = pd.read_csv(WORK / "phase4_test_metrics.csv"), pd.read_csv(WORK / "phase4_efficiency.csv"), \
    pd.read_csv(WORK / "phase4b_compute_benchmark.csv")
_alpha = CONFIG["significance"]["alpha"]
_p = _sig["phase4a"]["primary"]
_w, _b = _p["wilcoxon"], _p["bootstrap_sharpe"]
_nd, _fz = _sig["phase4b"]["selected_vs_daily"], _sig["phase4b"]["selected_vs_frozen_at_start"]
_rates = _sig["phase4b"]["test_rebuild_rate"]
_selN = f"{_winner['winner_arm']}_selected"

_4a_concl = ("is significantly better than" if (_w["p_value"] < _alpha and _b["ci_95"][0] > 0) else
             "is not shown to be significantly better than")
_4b_concl = ("non-inferior to daily rebuilding" if _nd["noninferiority"]["non_inferior"]
             else "NOT shown to be non-inferior to daily rebuilding")
L = []
L += ["# DMRGT — Phase 4 Results: Mamba encoder, adaptive training, regime-triggered graph refresh", "",
      "*Auto-generated from the run's JSON/CSV files. Every number below is read from a file, none typed by hand.*", "",
      "## 1. Objective", "",
      "Phase 4A: does a selective state-space (Mamba) encoder with adaptive training beat the Phase 3 Transformer "
      "encoder inside the same notebook, schedule and graph? Phase 4B: can the daily graph rebuild be replaced by "
      "an inference-time refresh triggered by a regime signal (correlation drift and/or the Mamba memory-reset "
      "signal Δ) with incremental correlation updates, without losing risk-adjusted performance and at lower "
      "graph-construction cost?", "",
      "## 2. Methodology Summary", "",
      f"- 4 walk-forward folds × {len(_seeds)} seeds = {_n_runs} paired runs per arm; identical folds to Phase 3.",
      f"- Arms: " + "; ".join(f"{a} = {CONFIG['arms'][a]['encoder']}, lookback {CONFIG['arms'][a]['lookback']}"
                              f"{', adaptive Huber' if CONFIG['arms'][a]['adaptive_huber'] else ''}" for a in _done_arms) + ".",
      f"- Batch size {_plan['batch_size']} for all arms; D skipped: {_plan.get('D_skipped_reason', 'no')}.",
      "- Selection (4A and 4B) on validation only; test split locked until both lock files existed; scored once.",
      f"- Config snapshot SHA-256: `{CONFIG_SHA256}`.",
      f"- Leakage gate: random-weights |IC| = {_gate['random_weights']['abs_ic_in_se']:.2f} SE, "
      f"shuffled-target |IC| = {_gate['shuffled_targets']['abs_ic_in_se']:.2f} SE (limit {_gate['max_abs_ic_in_standard_errors']} SE); "
      f"passed: {_gate['passed']}.", "",
      "## 3. Phase 4A — Validation Summary (selection)", "", _md(_val4a), "",
      f"**Winner: arm {_winner['winner_arm']}** (tiebreak applied: {_winner['tiebreak_applied']}).", "",
      "## 4. Test Results (mean over the 20 runs per model)", "", _md(_tm), "",
      "## 5. Significance Tests", "",
      f"### 5.1 Phase 4A primary — {_p['comparison']}", "",
      f"- Wilcoxon (paired rank IC): wins {_w['wins']}/{_w['n_pairs']}, p = {_w['p_value']:.4f}, "
      f"mean paired difference {_w['mean_paired_difference']:+.5f}.",
      f"- Stationary bootstrap (Sharpe difference): observed {_b['observed_sharpe_difference']:+.3f}, "
      f"95% CI [{_b['ci_95'][0]:+.3f}, {_b['ci_95'][1]:+.3f}], one-sided p = {_b['bootstrap_p_value']:.4f}.", ""]
for _k, _v in _sig["phase4a"].get("secondary", {}).items():
    L.append(f"- Secondary {_k}: Wilcoxon p = {_v['wilcoxon']['p_value']:.4f}; bootstrap CI "
             f"[{_v['bootstrap_sharpe']['ci_95'][0]:+.3f}, {_v['bootstrap_sharpe']['ci_95'][1]:+.3f}].")
for _k in ("secondary_holm_wilcoxon", "secondary_holm_bootstrap_two_sided"):
    if _k in _sig["phase4a"]:
        L.append(f"- Holm-adjusted ({_k}): " + "; ".join(f"{n}: p_adj = {v['p_adj']:.4f} (reject: {v['reject']})"
                                                         for n, v in _sig["phase4a"][_k].items()))
L += ["", f"### 5.2 Phase 4B — selected policy `{_pol['policy']['name']}`", "",
      f"- Validation: realised rebuild rate {_pol['realised_val_rebuild_rate']:.3f}, Sharpe "
      f"{_pol['val_sharpe_selected']:.3f} vs daily {_pol['val_sharpe_daily']:.3f} "
      f"(non-inferiority margin {_pol['val_margin']}; met: {_pol['validation_noninferiority_met']}).",
      f"- Test rebuild rates: " + ", ".join(f"{k} = {v:.3f}" for k, v in _rates.items()) + ".",
      f"- Non-inferiority vs daily: bootstrap 95% CI lower bound {_nd['noninferiority']['ci95_lower_bound']:+.3f} "
      f"vs −{_nd['noninferiority']['margin']} → non-inferior: {_nd['noninferiority']['non_inferior']}; "
      f"Wilcoxon on rank IC p = {_nd['wilcoxon']['p_value']:.4f}.",
      f"- Superiority vs frozen_at_start: Sharpe difference {_fz['bootstrap_sharpe']['observed_sharpe_difference']:+.3f}, "
      f"one-sided p = {_fz['bootstrap_sharpe']['bootstrap_p_value']:.4f}.", "",
      "### 5.3 Phase 4B validation table", "", _md(_val4b), "",
      "## 6. Efficiency and Technical Effect", "", "### 6.1 Model efficiency", "", _md(_eff), "",
      "### 6.2 Graph-construction cost (real 40-stock data, CPU, float64)", "",
      f"- Per update: full recompute {_real['full_recompute_ms_median']:.3f} ms vs incremental "
      f"{_real['incremental_update_ms_median']:.3f} ms; edge build {_real['edge_build_ms_median']:.3f} ms.",
      f"- Test-period total (fold 4): full-recompute daily {_real['test_period_total_seconds_full_recompute_daily']:.3f} s, "
      f"incremental daily {_real['test_period_total_seconds_incremental_daily']:.3f} s, "
      f"incremental + selected policy {_real['test_period_total_seconds_incremental_selected']:.3f} s.", "",
      "### 6.3 Synthetic scaling benchmark", "", _md(_bench), "",
      "## 7. Conclusion (rule-derived)", "",
      f"- Phase 4A: winner {_winner['winner_arm']} {_4a_concl} the Transformer arm A (alpha = {_alpha}).",
      f"- Phase 4B: selected policy `{_pol['policy']['name']}` is {_4b_concl} at margin "
      f"{_nd['noninferiority']['margin']} Sharpe, rebuilding on {_rates[_selN]:.1%} of test days.", "",
      "## 8. Notes on Interpretation", "",
      "- The model is trained on daily graphs; only the inference-time graph refresh changes in 4B (no per-policy retraining).",
      "- The backtest, cost model and Sharpe construction are the Phase 3 harness, unchanged.",
      "- Comparison with the Phase 3 Transformer is done locally after the run (spec §16).", "",
      "## 9. Reproducibility Record", "",
      f"- Run manifest: `run_manifest.json`; budget plan: `budget_plan.json` "
      f"(projected total {_plan['projection']['with_D' if _plan['run_D'] else 'without_D']['projected_total_h']:.2f} h).",
      f"- Winner checkpoint hashes: `phase4_winner.json`; test prediction hashes: `phase4_TEST_SCORED.json`.",
      f"- Code that produced the results: `dmrgt4/*.py` in this run's output.", ""]
(WORK / "PHASE4_RESULTS_DRAFT.md").write_text("\n".join(L))
print("wrote PHASE4_RESULTS_DRAFT.md")
print(f"total run time {(time.time() - RUN_T0) / 3600:.2f} h")
